# GLiNER multitask

> One GLiNER2 model trained on gold labels for every task its heads can do: decisions of any kind, entities of any type, relations between entities and between events (what causes what, what must happen first, what is part of what, what comes before what), and records such as an event with its arguments. Readers that turn public datasets into GLiNER2 training records with exact character offsets, a trainer for a full fine-tune or an adapter, and scores with confidence intervals.

In [ ]:
#| default_exp multitask

In [ ]:
#| export
import ast, bisect, copy, gzip, inspect, json, math, random, re, time, warnings, zlib
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from fastcore.basics import patch
from torch.utils.data import DataLoader

from sysone.distill import (BASE, DECIDE, DistillLearner, _Offsets, _unwrap, _unique, _human, _words, _labels_of, _norm, _f1, _softmax, ece,
                            ask_schema, row_length, epoch_batches, decision_logits, length_batches, lora_targets, distinct_tasks, sample_types)

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, tempfile
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*attn_implementation.*")
warnings.filterwarnings("ignore", message=r".*torch\.jit\.script.*")
from sysone.distill import load_extractor
from itertools import combinations
from gliner2 import AutoExtractor
from peft import PeftModel
FIX = Path("fixtures/tiny-gliner2")

A GLiNER2 model reads a text together with a *schema*, the things to find in it, and has a head for each kind of thing:

- **Entities**: spans of a type you name ("person", "event", "claim").
- **Relations**: pairs of spans, a head and a tail, joined by a relation you name ("causes", "country").
- **Records** (GLiNER2 calls them structures): a group of fields filled from the text, such as an event with its trigger word, its agent and its place.
- **Decisions** (classifications): a label chosen from the labels you give, for a question you can phrase.

[gliner2.5-multi-v1](https://huggingface.co/fastino/gliner2.5-multi-v1) does all four in about a hundred languages. This notebook builds what it takes to train such a model further on gold labels, the answers people wrote, from many public datasets at once, so that one model gets better at all four. The [tutorial](tutorials/12_system_one_multi.ipynb) runs it at scale; here every piece is tested on a tiny model in `fixtures/tiny-gliner2`.

The pieces need sysone's `gliner` extra (`pip install "sysone[gliner]"`).

::: {.callout-tip title="Jargon"}
- **Gold label**: the answer a dataset's annotators gave; a model is trained to give it too.
- **Span**: a stretch of the text, given by its first character and the one after its last (*offsets*).
- **Trigger**: the word that names an event ("attacked", "flood", "elections").
- **Full fine-tune** and **adapter**: training every weight of the model, or small added matrices beside its layers (LoRA) while the model's own weights stay as they are.
:::

## Spans

A dataset gives a span by its offsets, by its words, or only by its text. GLiNER2 2.1 takes a span as `{"text", "start", "end"}`, which says *which* occurrence is meant: in "the storm hit, then a second storm came", the two storms are different spans. Every reader below produces spans this way.

In [ ]:
#| export
def span(text:str, start:int, end:int) -> dict:
    "A gold span: the characters of `text` from `start` to `end` (exclusive), and those offsets"
    return {"text": text[start:end], "start": int(start), "end": int(end)}

def _splitter():
    from gliner2.processing.word_splitter import WhitespaceTokenSplitter
    return WhitespaceTokenSplitter()

class Words:
    "The words GLiNER2's default splitter reads in a text, to widen a span to the whole words it touches"
    def __init__(self, text:str):
        self.text = text
        ws = [(s, e) for _, s, e in _splitter()(text, lower=False)]
        self.starts, self.ends = [s for s, _ in ws], [e for _, e in ws]
    def __call__(self, start:int, end:int, grow:float=3.0) -> dict|None:
        "The span of the words `start`–`end` touches; None when it touches none, or when they are more than `grow` times longer"
        i, j = bisect.bisect_right(self.ends, start), bisect.bisect_left(self.starts, end) - 1
        if i > j: return None
        s, e = self.starts[i], self.ends[j]
        if e - s > grow * max(1, end - start) and e - s - (end - start) > 4: return None
        return span(self.text, s, e)

GLiNER2 reads words, not characters, so a span that starts or ends inside a word stands for the whole word. Some datasets mark part of a word: RED^FM tags "japan" inside the German "japanischen". `Words` widens such a span the way GLiNER2 will read it, so training and scoring agree. Its default splitter keeps a run of letters together, Chinese and Japanese characters included, so in text without spaces a span would widen to a whole run of characters; `Words` refuses those (None), and the readers drop them.

In [ ]:
t = "Die Nikon D1 kommt vom japanischen Hersteller Nikon."
w = Words(t)
test_eq(w(t.index("japan"), t.index("japan") + 5)["text"], "japanischen")
test_eq(w(4, 12), span(t, 4, 12))                                  # whole words stay as they are
test_eq(w(3, 13)["text"], "Nikon D1")                              # spaces around a span are dropped
c = "苹果公司总部位于加利福尼亚州库比蒂诺。"
test_eq(Words(c)(0, 4), None)                                      # a run of CJK characters is one word to GLiNER2

Corpora that were tokenized before release give a span by token positions, and their tokens do not always match the text: MAVEN lowercases some sentences' tokens, and writes quotes as ` `` ` and `''`. `token_offsets` finds each token in the text, in order.

In [ ]:
#| export
QUOTES = {"``": '"“”«„', "''": '"”“»'}     # the tokens corpora write for quotes, and the characters the text may have
BRACKETS = {"-LRB-": "(", "-RRB-": ")", "-LSB-": "[", "-RSB-": "]", "-LCB-": "{", "-RCB-": "}"}

def token_offsets(tokens:list, text:str, window:int=40) -> list:
    "Each token's (start, end) in `text`, matched left to right ignoring case (`` and '' match a quote); None for one not found within `window` characters"
    out, pos = [], 0
    for t in tokens:
        best = None
        for c in [t, *QUOTES.get(t, ""), *([BRACKETS[t]] if t in BRACKETS else [])]:
            m = re.compile(re.escape(c), re.I).search(text, pos, min(len(text), pos + window + len(c)))
            if m and (best is None or m.start() < best[0]): best = (m.start(), m.end())
        out.append(best)
        if best: pos = best[1]
    return out

In [ ]:
sent = 'A corps led by Giuseppe Garibaldi sailed from Quarto ("Spedizione dei Mille").'
toks = ["a", "corps", "led", "by", "giuseppe", "garibaldi", "sailed", "from", "quarto", "(", "``", "Spedizione", "dei", "Mille", "''", ")", "."]
offs = token_offsets(toks, sent)
test_eq([sent[a:b] for a, b in offs[4:6]], ["Giuseppe", "Garibaldi"])
test_eq([sent[a:b] for a, b in offs[9:16]], ["(", '"', "Spedizione", "dei", "Mille", '"', ")"])
test_eq(token_offsets(["missing"], "nothing here"), [None])

## Gold rows and training records

A *row* is one text and what to find in it. It has the shape the [distillation notebook](32_gliner_distill.ipynb) uses, with gold spans as offsets:

```python
{"id": "redfm-de/0", "kind": "extraction", "source": "redfm-de", "lang": "de", "text": "Die Nikon D1 …",
 "ask":  {"entities": ["organization", "location"],            # a list, or {name: description}
          "relations": {"manufacturer": "who makes it"},
          "structures": {"Attack": {"trigger": "the word naming the attack", "attacker": ""}},
          "classifications": [{"task": "sentiment", "labels": ["positive", "negative"], "prompt": "How does the writer feel?"}]},
 "gold": {"entities": {"organization": [{"text": "Nikon", "start": 46, "end": 51}]},
          "relations": {"manufacturer": [[head span, tail span]]},
          "structures": {"Attack": [{"trigger": span, "attacker": [spans]}]},
          "classifications": {"sentiment": "positive"}}}            # a label, or {label: probability}
```

`gold_example` turns a row into GLiNER2's training record. A type, relation or record that the row asks for and its gold lacks becomes a negative: the model learns that the text has none. A decision whose gold is a distribution, as when annotators disagreed, keeps it as soft targets.

In [ ]:
#| export
def _span_list(x) -> list:
    "A gold value as a list of spans: a span, a string, or a list of them"
    return list(x) if isinstance(x, list) else [] if x in (None, "") else [x]

def _described(d:dict) -> dict:
    "Descriptions without the empty ones: {name: text}, or for records {record: {field: text}}"
    out = {k: (_described(v) if isinstance(v, dict) else v) for k, v in d.items()}
    return {k: v for k, v in out.items() if v}

def gold_example(row:dict) -> dict:
    "A GLiNER2 training record from a row's ask and gold: spans keep their offsets, what is asked and absent is a negative, a decision's distribution travels beside its labels as soft targets"
    sd = ask_schema(row["ask"]).build(); gold = row.get("gold") or {}
    out, absent, soft = {k: d for k, v in sd.items() if k.endswith("descriptions") and (d := _described(v))}, [], {}
    if sd.get("json_structures"):
        js = []
        for item in sd["json_structures"]:
            (name, fields), = item.items()
            insts = (gold.get("structures") or {}).get(name) or []
            js += [{name: {f: inst.get(f) or "" for f in fields}} for inst in insts] or [{name: {f: "" for f in fields}}]
        out["json_structures"] = js
    if sd.get("entities"):
        ents = gold.get("entities") or {}
        out["entities"] = {t: _span_list(ents.get(t)) for t in sd["entities"]}
    if sd.get("relations"):
        rels = []
        for item in sd["relations"]:
            (name, _), = item.items()
            pairs = (gold.get("relations") or {}).get(name) or []
            rels += [{name: {"head": h, "tail": t}} for h, t in pairs] or [{name: {"head": "", "tail": ""}}]
            if not pairs: absent.append(name)
        out["relations"] = rels
    cls = []
    for c in sd.get("classifications") or []:
        g = (gold.get("classifications") or {}).get(c["task"])
        if g is None: continue                                     # a question without gold teaches nothing
        p = g if isinstance(g, dict) else {l: float(l == g) for l in c["labels"]}
        cls.append({**{k: v for k, v in c.items() if k != "true_label"}, "true_label": [max(c["labels"], key=lambda l: p.get(l, 0.0))]})
        if isinstance(g, dict): soft[c["task"]] = {l: float(p.get(l, 0.0)) for l in c["labels"]}
    if cls: out["classifications"] = cls
    return {"id": row.get("id"), "kind": row.get("kind"), "source": row.get("source"), "input": row["text"], "output": out,
            "absent_relations": absent, "soft": soft, "length": row_length(row)}

def _all_spans(x):
    if isinstance(x, dict) and {"text", "start", "end"} <= x.keys(): yield x
    elif isinstance(x, dict): yield from (s for v in x.values() for s in _all_spans(v))
    elif isinstance(x, list): yield from (s for v in x for s in _all_spans(v))

def spans_ok(e:dict) -> bool:
    "Whether every gold span of a training record is its text's characters at its offsets"
    return all(e["input"][s["start"]:s["end"]] == s["text"] for s in _all_spans(e["output"]))

In [ ]:
text = "The storm hit the coast and the flood destroyed the bridge. The storm ended on Monday."
s = lambda w, k=0: span(text, i := [m.start() for m in re.finditer(w, text)][k], i + len(w))
row = {"id": "t/0", "kind": "mixed", "source": "test", "text": text,
       "ask": {"entities": {"event": "something that happens", "person": ""}, "relations": ["causes", "happens before"],
               "structures": {"Destroying": {"trigger": "the word naming it", "Patient": ""}},
               "classifications": [{"task": "tone", "labels": ["calm", "alarming"], "prompt": "How does it read?"}]},
       "gold": {"entities": {"event": [s("storm"), s("flood"), s("destroyed"), s("storm", 1), s("ended")]},
                "relations": {"causes": [[s("storm"), s("flood")], [s("flood"), s("destroyed")]]},
                "structures": {"Destroying": [{"trigger": s("destroyed"), "Patient": [s("the bridge")]}]},
                "classifications": {"tone": {"calm": 0.3, "alarming": 0.7}}}}
ex = gold_example(row)
test_eq(ex["output"]["entities"]["person"], [])                    # asked and absent: a negative
test_eq(ex["absent_relations"], ["happens before"])
test_eq(ex["output"]["classifications"][0]["true_label"], ["alarming"])
test_eq(ex["soft"], {"tone": {"calm": 0.3, "alarming": 0.7}})
test_eq(ex["output"]["entity_descriptions"], {"event": "something that happens"})
assert spans_ok(ex) and not spans_ok({"input": "abc", "output": {"entities": {"x": [{"text": "b", "start": 0, "end": 1}]}}})
ex["output"]

{'json_descriptions': {'Destroying': {'trigger': 'the word naming it'}},
 'entity_descriptions': {'event': 'something that happens'},
 'json_structures': [{'Destroying': {'trigger': {'text': 'destroyed',
     'start': 38,
     'end': 47},
    'Patient': [{'text': 'the bridge', 'start': 48, 'end': 58}]}}],
 'entities': {'event': [{'text': 'storm', 'start': 4, 'end': 9},
   {'text': 'flood', 'start': 32, 'end': 37},
   {'text': 'destroyed', 'start': 38, 'end': 47},
   {'text': 'storm', 'start': 64, 'end': 69},
   {'text': 'ended', 'start': 70, 'end': 75}],
  'person': []},
 'relations': [{'causes': {'head': {'text': 'storm', 'start': 4, 'end': 9},
    'tail': {'text': 'flood', 'start': 32, 'end': 37}}},
  {'causes': {'head': {'text': 'flood', 'start': 32, 'end': 37},
    'tail': {'text': 'destroyed', 'start': 38, 'end': 47}}},
  {'happens before': {'head': '', 'tail': ''}}],
 'classifications': [{'task': 'tone',
   'labels': ['calm', 'alarming'],
   'multi_label': False,
   'cls_threshol

## Decisions from tasksource-jev

[tasksource-jev-typed-decisions](https://huggingface.co/datasets/tasksource/tasksource-jev-typed-decisions) gathers about a million decisions from hundreds of tasks, each a text (`state`), a `question`, and a `target` over `options`. A question is one of three kinds, the same three sysone's typed decisions have:

- `choice`: pick one of the options; the target is a distribution over them.
- `score`: an ordered scale, lowest first; the target is a distribution over the levels.
- `noul`: a yes/no question; the target is the probability of yes.

Questions about the same text share a `group_id`. `jev_requests` gathers them into one request, as the dataset's own notes say to, and `jev_decision_row` asks all of them in one GLiNER2 row, labelled the way sysone's GLiNER backend labels a typed question: a choice's options as its labels, a score's levels as `0`, `1`, … with their descriptions, a noul as `yes`/`no`. The targets become soft gold.

In [ ]:
#| export
def jev_requests(rows) -> list:
    "jev rows (one question each, in stored order) gathered into requests: the questions that share a group and a state, with their targets"
    out, cur, key = [], None, None
    for r in rows:
        k = (r["group_id"], r["state"])
        if k != key:
            if cur: out.append(cur)
            cur, key = {"id": r["group_id"], "state": r["state"], "source": r["source"], "variant": r.get("variant"), "questions": {}, "targets": {}}, k
        cur["questions"][r["question_id"]] = {"type": r["kind"], "instructions": r["question"]} | ({"criteria": list(r["options"])} if len(r["options"]) else {})
        cur["targets"][r["question_id"]] = [float(x) for x in r["target"]]
    return out + ([cur] if cur else [])

def jev_lang(source:str) -> str|None:
    "A jev source's language: the code ending a `multilingual/…` source's name, English otherwise"
    if not source.startswith("multilingual/"): return "en"
    code = source.rsplit("/", 1)[-1]
    return code.split("-", 1)[1] if code.startswith("translation-") else code

def jev_decision_row(req:dict, id:str|None=None, suite:str|None=None) -> dict|None:
    "A row asking every question of a jev request, one GLiNER2 head each, labelled as sysone's GLiNER backend labels typed questions, the targets as soft gold; None when a question cannot be asked"
    from sysone.gliner import gliner_tasks
    try: tasks = gliner_tasks(req["questions"])
    except (ValueError, KeyError, TypeError): return None
    rename = distinct_tasks([t["task"] for t in tasks])
    cls, gold = [], {}
    for t in tasks:
        q, target, names = req["questions"][t["task"]], req["targets"][t["task"]], list(t["names"])
        if len(set(names)) != len(names) or not names: return None
        if q["type"] == "noul":
            if len(target) != 1: return None
            p = {"yes": target[0], "no": 1.0 - target[0]}
        elif len(target) != len(names): return None
        else: p = dict(zip(names, target))
        c = {"task": rename[t["task"]], "labels": names}
        if isinstance(t["labels"], dict) and any(t["labels"].values()): c["label_descriptions"] = {l: d for l, d in t["labels"].items() if d}
        if (t["instruction"] or "").strip(): c["prompt"] = t["instruction"].strip()
        cls.append(c); gold[c["task"]] = p
    row = {"id": id or req["id"], "kind": "decision", "source": req["source"], "lang": jev_lang(req["source"]), "text": req["state"],
           "ask": {"classifications": cls}, "gold": {"classifications": gold}}
    return row | ({"suite": suite} if suite else {})

In [ ]:
jev = [{"state": "The battery died after two days.", "kind": "choice", "options": ["praise", "complaint", "question"], "target": [0.0, 0.9, 0.1],
        "question": "What is the customer doing?", "source": "reviews", "group_id": "g1", "question_id": "intent"},
       {"state": "The battery died after two days.", "kind": "score", "options": ["calm", "annoyed", "furious"], "target": [0.2, 0.8, 0.0],
        "question": "How upset is the customer?", "source": "reviews", "group_id": "g1", "question_id": "anger"},
       {"state": "Il pleut à Paris.", "kind": "noul", "options": [], "target": [0.95], "question": "Is it about the weather?",
        "source": "multilingual/xnli/fr", "group_id": "g2", "question_id": "about-weather"}]
reqs = jev_requests(jev)
test_eq([len(r["questions"]) for r in reqs], [2, 1])
rows = [jev_decision_row(r) for r in reqs]
test_eq([c["labels"] for c in rows[0]["ask"]["classifications"]], [["praise", "complaint", "question"], ["0", "1", "2"]])
test_eq(rows[0]["ask"]["classifications"][1]["label_descriptions"], {"0": "calm", "1": "annoyed", "2": "furious"})
test_eq(rows[1]["gold"]["classifications"]["about-weather"], {"yes": 0.95, "no": 0.050000000000000044})
test_eq(rows[1]["lang"], "fr")
test_eq(gold_example(rows[0])["output"]["classifications"][1]["true_label"], ["1"])
rows[0]["ask"]

{'classifications': [{'task': 'intent',
   'labels': ['praise', 'complaint', 'question'],
   'prompt': 'What is the customer doing?'},
  {'task': 'anger',
   'labels': ['0', '1', '2'],
   'label_descriptions': {'0': 'calm', '1': 'annoyed', '2': 'furious'},
   'prompt': 'How upset is the customer?'}]}

## Spans from tasksource-jev

The dataset's `extraction` config asks for spans: every claim of an abstract, every noun phrase, every library building, the answer to a question about a passage. Each row has several typed `fields` (`list`: every span of a kind; `text`: one span answering a question) and an answer per field, as offsets or null.

A row whose fields are all lists asks for kinds of span, which is what GLiNER2's entity head does, so its fields become entity types, each with its instruction as the description. A row with a question among its fields becomes one record, a field per question.

In [ ]:
#| export
def _names(fields:list) -> list:
    "Field names made readable, kept distinct"
    seen, out = Counter(), []
    for f in fields:
        n = _human(f["name"]).lower(); seen[n] += 1
        out.append(n if seen[n] == 1 else f"{n} {seen[n]}")
    return out

def jev_extraction_row(r:dict, suite:str|None=None) -> dict|None:
    "A row from jev's extraction config: list fields as entity types (their instructions as descriptions); a row with a question among its fields as one record, a field per question"
    text, names = r["state"], _names(r["fields"])
    vals = [[span(text, v["start"], v["end"]) for v in (a.get("values") if a.get("values") is not None else [])] if not a.get("null") else []
            for a in r["answers"]]
    if any(text[s["start"]:s["end"]] != s["text"] for v in vals for s in v) or len(vals) != len(names): return None
    if all(f["type"] == "list" for f in r["fields"]):
        ask = {"entities": {n: f["instruction"] for n, f in zip(names, r["fields"])}}
        gold = {"entities": {n: v for n, v in zip(names, vals)}}
    else:
        name = _human(r["source"].split("/")[0]).lower()
        ask = {"structures": {name: {n: {"description": f["instruction"], "dtype": "str" if f["type"] == "text" else "list"} for n, f in zip(names, r["fields"])}}}
        inst = {n: (v[:1] if f["type"] == "text" else v) for n, f, v in zip(names, r["fields"], vals)}  # a question's first gold span; the rest are alternatives
        gold = {"structures": {name: [inst] if any(inst.values()) else []}}
    row = {"id": r["id"], "kind": "extraction", "source": r["source"], "lang": None, "text": text, "ask": ask, "gold": gold}
    return row | ({"suite": suite} if suite else {})

In [ ]:
ner = {"id": "few-nerd/0", "source": "few-nerd/supervised", "state": "Ridwan 's forces were defeated in the Battle of Tell Aghdi in 1105 .",
       "fields": [{"name": "library_building", "type": "list", "instruction": "Every library (building) mentioned in the text, as written."},
                  {"name": "attack_battle_war_or_military_conflict_event", "type": "list", "instruction": "Every battle mentioned in the text, as written."}],
       "answers": [{"null": True, "values": []}, {"null": False, "values": [{"text": "the Battle of Tell Aghdi", "start": 34, "end": 58}]}]}
qa = {"id": "squad/0", "source": "squad_v2", "state": "Normandy is a region in France. The Normans gave their name to it.",
      "fields": [{"name": "q1", "type": "text", "instruction": "In what country is Normandy?"}, {"name": "q2", "type": "text", "instruction": "Who founded Rome?"}],
      "answers": [{"null": False, "values": [{"text": "France", "start": 24, "end": 30}]}, {"null": True, "values": []}]}
a, b = jev_extraction_row(ner), jev_extraction_row(qa)
test_eq(list(a["ask"]["entities"]), ["library building", "attack battle war or military conflict event"])
test_eq(a["gold"]["entities"]["attack battle war or military conflict event"][0]["text"], "the Battle of Tell Aghdi")
test_eq(b["gold"]["structures"]["squad v2"][0]["q1"][0]["text"], "France")
test_eq(b["ask"]["structures"]["squad v2"]["q2"]["description"], "Who founded Rome?")

## Relations between entities

Two datasets with human-checked relations, in two shapes:

- [RED^FM](https://huggingface.co/datasets/Babelscape/REDFM) (Arabic, Chinese, English, French, German, Italian, Spanish) and its larger automatic sibling SRED^FM (18 languages): Wikipedia passages, entities as offsets, relations named by their Wikidata property ("country", "manufacturer", "place of birth").
- [Re-DocRED](https://huggingface.co/datasets/tonytan48/Re-DocRED) (English): whole Wikipedia introductions, tokenized, entities with all their mentions, relations between entities by Wikidata property code.

A row asks for the relation types its text has and, as negatives, some it lacks, drawn from all the types of the dataset.

In [ ]:
#| export
REDFM_TYPES = {"PER": "person", "ORG": "organization", "LOC": "location", "MISC": "other", "DATE": "date", "TIME": "time", "NUMBER": "number",
               "Concept": "concept"}

def redfm_row(r:dict, source:str, relation_pool:list, rng:random.Random, absent:float=1/3, suite:str|None=None) -> dict|None:
    "A row from RED^FM or SRED^FM: its entities by type and its relations by property name, spans widened to whole words; about `absent` of the relation types asked are ones the text lacks"
    text, w = r["text"], Words(r["text"])
    spans, ents = {}, defaultdict(list)
    for e in r["entities"]:
        b = e.get("boundaries")
        if not b or (s := w(*b)) is None: continue
        spans[tuple(b)] = s
        t = REDFM_TYPES.get(e.get("type"))
        if t and s not in ents[t]: ents[t].append(s)
    rels = defaultdict(list)
    for x in r["relations"]:
        h, t = spans.get(tuple(x["subject"].get("boundaries") or ())), spans.get(tuple(x["object"].get("boundaries") or ()))
        name = ((x.get("predicate") or {}).get("surfaceform") or "").strip().lower()
        if h and t and name and h != t and [h, t] not in rels[name]: rels[name].append([h, t])
    if not rels: return None
    names = sample_types(list(rels), relation_pool, rng, absent)
    types = sample_types(list(ents), list(REDFM_TYPES.values()), rng, absent) if ents else []
    row = {"id": f"{source}/{r.get('docid', '')}", "kind": "extraction", "source": source, "lang": r.get("lan"), "text": text,
           "ask": {"relations": names} | ({"entities": types} if types else {}),
           "gold": {"relations": {n: rels[n] for n in names if n in rels}, "entities": {t: ents[t] for t in types if t in ents}}}
    return row | ({"suite": suite} if suite else {})

DOCRED_TYPES = {"PER": "person", "ORG": "organization", "LOC": "location", "TIME": "time", "NUM": "number", "MISC": "other"}

def joined(sents:list) -> tuple:
    "Tokenized sentences joined by single spaces: the text, and each sentence's tokens' (start, end)"
    parts, offs, pos = [], [], 0
    for sent in sents:
        o = []
        for tok in sent:
            if pos: parts.append(" "); pos += 1
            o.append((pos, pos + len(tok))); parts.append(tok); pos += len(tok)
        offs.append(o)
    return "".join(parts), offs

def windows(lengths:list, most:int) -> list:
    "Consecutive sentences grouped into windows of up to `most` words; a longer sentence is a window of its own"
    out, cur, n = [], [], 0
    for i, k in enumerate(lengths):
        if cur and n + k > most: out.append(cur); cur, n = [], 0
        cur.append(i); n += k
    return out + ([cur] if cur else [])

def _closest(heads:list, tails:list) -> list|None:
    "The head and tail mentions nearest each other: the pair that stands for a relation between two entities with several mentions"
    pairs = [(h, t) for h in heads for t in tails if h != t]
    return list(min(pairs, key=lambda p: abs(p[0]["start"] - p[1]["start"]))) if pairs else None

def _labels(x) -> list:
    "DocRED's relation labels as dicts (h, t, r), from a list of dicts, its string, or the columns of the Hub's loader"
    if isinstance(x, str): x = ast.literal_eval(x)
    if isinstance(x, dict): return [{"h": h, "t": t, "r": r} for h, t, r in zip(x["head"], x["tail"], x["relation_id"])]
    return list(x)

def docred_rows(doc:dict, rel_names:dict, relation_pool:list, rng:random.Random, most:int=250, absent:float=1/3, source:str="re-docred",
                suite:str|None=None) -> list:
    "Rows from a (Re-)DocRED document, a window of sentences each: the mentions by type, and each relation whose entities both have a mention in the window (the nearest two standing for them)"
    vs = ast.literal_eval(doc["vertexSet"]) if isinstance(doc["vertexSet"], str) else doc["vertexSet"]
    sents = list(doc["sents"]); text, offs = joined(sents)
    ment = lambda m: (offs[m["sent_id"]][m["pos"][0]][0], offs[m["sent_id"]][m["pos"][1] - 1][1])
    out = []
    for k, win in enumerate(windows([len(s) for s in sents], most)):
        lo, hi = offs[win[0]][0][0], offs[win[-1]][-1][1]; sub = text[lo:hi]
        inside = [[span(sub, a - lo, b - lo) for m in v if m["sent_id"] in win for a, b in [ment(m)]] for v in vs]
        ents = defaultdict(list)
        for v, ms in zip(vs, inside):
            for s in ms:
                if s not in ents[DOCRED_TYPES.get(v[0]["type"], "other")]: ents[DOCRED_TYPES.get(v[0]["type"], "other")].append(s)
        rels = defaultdict(list)
        for l in _labels(doc["labels"]):
            p = _closest(inside[l["h"]], inside[l["t"]]); name = rel_names.get(l["r"], l["r"])
            if p and p not in rels[name]: rels[name].append(p)
        if not rels: continue
        names, types = sample_types(list(rels), relation_pool, rng, absent), list(ents)
        out.append({"id": f"{source}/{doc.get('title', '')}/{k}", "kind": "extraction", "source": source, "lang": "en", "text": sub,
                    "ask": {"relations": names, "entities": types},
                    "gold": {"relations": {n: rels[n] for n in names if n in rels}, "entities": dict(ents)}} | ({"suite": suite} if suite else {}))
    return out

In [ ]:
red = {"docid": "3233638-0", "lan": "de",
       "text": "Die Nikon D1 ist eine digitale Spiegelreflexkamera des japanischen Herstellers Nikon, die im Oktober 1999 in den Markt eingeführt wurde.",
       "entities": [{"boundaries": [4, 12], "surfaceform": "Nikon D1", "type": "MISC"}, {"boundaries": [55, 60], "surfaceform": "japan", "type": "LOC"},
                    {"boundaries": [79, 84], "surfaceform": "Nikon", "type": "ORG"}],
       "relations": [{"subject": {"boundaries": [4, 12]}, "predicate": {"surfaceform": "manufacturer"}, "object": {"boundaries": [79, 84]}},
                     {"subject": {"boundaries": [79, 84]}, "predicate": {"surfaceform": "country"}, "object": {"boundaries": [55, 60]}}]}
r = redfm_row(red, "redfm-de", ["manufacturer", "country", "place of birth", "author", "member of"], random.Random(0))
test_eq(r["gold"]["relations"]["country"][0][1]["text"], "japanischen")
assert {"manufacturer", "country"} <= set(r["ask"]["relations"]) and len(r["ask"]["relations"]) == 3
assert spans_ok(gold_example(r))

doc = {"title": "Loud Tour", "sents": [["The", "Loud", "Tour", "was", "a", "tour", "by", "Rihanna", "."], ["Rihanna", "is", "from", "Barbados", "."]],
       "vertexSet": [[{"name": "Loud Tour", "pos": [1, 3], "sent_id": 0, "type": "MISC"}],
                     [{"name": "Rihanna", "pos": [7, 8], "sent_id": 0, "type": "PER"}, {"name": "Rihanna", "pos": [0, 1], "sent_id": 1, "type": "PER"}],
                     [{"name": "Barbados", "pos": [3, 4], "sent_id": 1, "type": "LOC"}]],
       "labels": "[{'evidence': [0], 'h': 0, 'r': 'P175', 't': 1}, {'evidence': [1], 'h': 1, 'r': 'P27', 't': 2}]"}
d = docred_rows(doc, {"P175": "performer", "P27": "country of citizenship"}, ["performer", "country of citizenship", "author"], random.Random(0))
test_eq(len(d), 1)
test_eq([(h["text"], t["text"], h["start"]) for h, t in d[0]["gold"]["relations"]["country of citizenship"]], [("Rihanna", "Barbados", 38)])  # the second Rihanna, nearer Barbados
test_eq(len(docred_rows(doc, {}, ["P175", "P27"], random.Random(0), most=6)), 2)   # one window a sentence: each keeps the relation inside it
assert all(spans_ok(gold_example(x)) for x in d)

## Events and how they relate

[MAVEN-ERE](https://github.com/THU-KEG/MAVEN-ERE) annotates 4,480 English Wikipedia articles with their events and the relations between them:

- **Causal**: one event *causes* another, or is a *precondition* of it, so that the second would not happen without the first.
- **Subevent**: one event is *part of* another, as a battle is part of a war.
- **Temporal**: *before*, *overlaps*, *contains* (in time), *simultaneous*, *begins on*, *ends on*, between events and dates.
- **Coreference**: two mentions of the same event.

A relation joins two *events*, and an event may be mentioned several times, so a row stands for it by the mentions nearest each other in its window of sentences. Each relation type gets a name GLiNER2 is asked for and a description of what it means; the event mentions and time expressions are its entities. A row asks every type, or, with `absent`, the types its window has and a few it lacks: ten relation types and thirty pairs in one row turned out to be more than one epoch of training could teach.

In [ ]:
#| export
EVENT_RELATIONS = {     # MAVEN-ERE's relation types: the name GLiNER2 is asked for, and its description
    "CAUSE": ("causes", "the first event brings about the second"),
    "PRECONDITION": ("is a precondition of", "the first event makes the second possible: without it, the second would not happen"),
    "SUBEVENT": ("has subevent", "the second event is a part of the first"),
    "BEFORE": ("happens before", "the first ends before the second begins"),
    "OVERLAP": ("overlaps in time with", "the two happen partly at the same time"),
    "CONTAINS": ("contains in time", "the second happens within the time of the first"),
    "SIMULTANEOUS": ("happens at the same time as", "the two happen over the same time"),
    "BEGINS-ON": ("begins when", "the two begin at the same time"),
    "ENDS-ON": ("ends when", "the two end at the same time"),
    "COREFERENCE": ("is the same event as", "both mentions refer to one event")}
EVENT_FAMILIES = {"causal": ["CAUSE", "PRECONDITION"], "subevent": ["SUBEVENT"],
                  "temporal": ["BEFORE", "OVERLAP", "CONTAINS", "SIMULTANEOUS", "BEGINS-ON", "ENDS-ON"], "coreference": ["COREFERENCE"]}
EVENT_ENTITIES = {"event": "something that happens or is done, named by the word that says so", "time expression": "a date, time, duration or frequency"}

def relation_families() -> dict:
    "Each relation's family: causal, subevent, temporal or coreference between events; logical between statements (implies, rules out)"
    return {EVENT_RELATIONS[k][0]: f for f, ks in EVENT_FAMILIES.items() for k in ks} | {"implies": "logical", "rules out": "logical"}

def maven_parse(doc:dict) -> tuple:
    "A MAVEN document's text (its sentences joined by spaces), each event's and time expression's mentions as (sentence, span), their types, and the related pairs by MAVEN-ERE type"
    sents = list(doc["sentences"]); text = " ".join(sents)
    base = [0]
    for s in sents[:-1]: base.append(base[-1] + len(s) + 1)
    offs = [[None if o is None else (o[0] + b, o[1] + b) for o in token_offsets(toks, sent)] for b, toks, sent in zip(base, doc["tokens"], sents)]
    def mention(sid, o):
        a, z = offs[sid][o[0]], offs[sid][o[1] - 1]
        return span(text, a[0], z[1]) if a and z and z[1] > a[0] else None
    mentions, types = {}, {}
    for ev in doc.get("events") or []:
        mentions[ev["id"]] = [(m["sent_id"], s) for m in ev["mention"] if (s := mention(m["sent_id"], m["offset"]))]
        types[ev["id"]] = ev["type"]
    for tx in doc.get("TIMEX") or []:
        s = mention(tx["sent_id"], tx["offset"]); mentions[tx["id"]] = [(tx["sent_id"], s)] if s else []
    pairs = defaultdict(list)
    for group in ("temporal_relations", "causal_relations"):
        for k, ps in (doc.get(group) or {}).items(): pairs[k] += [tuple(p) for p in ps]
    pairs["SUBEVENT"] += [tuple(p) for p in doc.get("subevent_relations") or []]
    return text, base, mentions, types, dict(pairs)

def _window_spans(text, base, sents, win, mentions):
    lo, hi = base[win[0]], base[win[-1]] + len(sents[win[-1]])
    sub, ws = text[lo:hi], set(win)
    return sub, {i: [span(sub, s["start"] - lo, s["end"] - lo) for sid, s in ms if sid in ws] for i, ms in mentions.items()}

def maven_ere_rows(doc:dict, most:int=250, per_type:int=16, rng:random.Random|None=None, families:list|None=None, source:str="maven-ere",
                   suite:str|None=None, absent:int|None=None) -> list:
    "Rows from a MAVEN-ERE document, a window of sentences each: its event mentions and time expressions, and the relations of `families` between them with both ends in the window; at most `per_type` pairs of a type. Each row asks every type of `families`, or with `absent` the types its window has and up to `absent` it lacks"
    rng = rng or random.Random(0)
    text, base, mentions, _, pairs = maven_parse(doc)
    asked = {EVENT_RELATIONS[k][0]: EVENT_RELATIONS[k][1] for f in (families or list(EVENT_FAMILIES)) for k in EVENT_FAMILIES[f]}
    out = []
    for n, win in enumerate(windows([len(t) for t in doc["tokens"]], most)):
        sub, inside = _window_spans(text, base, doc["sentences"], win, mentions)
        events = _unique_spans(s for i, ss in inside.items() if i.startswith("EVENT") for s in ss)
        times = _unique_spans(s for i, ss in inside.items() if i.startswith("TIME") for s in ss)
        if not events: continue
        rels = {}
        for k, ps in pairs.items():
            found = [p for a, b in ps if (p := _closest(inside.get(a) or [], inside.get(b) or []))]
            if len(found) > per_type: found = rng.sample(found, per_type)
            if found: rels[EVENT_RELATIONS[k][0]] = found
        coref = [[a, b] for i, ss in inside.items() if i.startswith("EVENT") for a, b in zip(ss, ss[1:])]
        if coref: rels[EVENT_RELATIONS["COREFERENCE"][0]] = coref[:per_type]
        ents = {"event": events} | ({"time expression": times} if times else {})
        names = list(asked)
        if absent is not None:
            lacking = [k for k in asked if k not in rels]
            names = [k for k in asked if k in rels] + rng.sample(lacking, min(absent, len(lacking))); rng.shuffle(names)
        out.append({"id": f"{source}/{doc.get('id', '')}/{n}", "kind": "extraction", "source": source, "lang": "en", "text": sub,
                    "ask": {"entities": {k: EVENT_ENTITIES[k] for k in ents}, "relations": {k: asked[k] for k in names}},
                    "gold": {"entities": ents, "relations": {k: v for k, v in rels.items() if k in names}}} | ({"suite": suite} if suite else {}))
    return out

def _unique_spans(spans) -> list:
    out = []
    for s in spans:
        if s not in out: out.append(s)
    return out

def maven_event_rows(doc:dict, type_pool:list, rng:random.Random, most:int=250, k:int=8, absent:float=1/3, source:str="maven-events",
                     suite:str|None=None) -> list:
    "Rows asking for a MAVEN document's events by type (168 types such as attack, arriving, causation), a window of sentences each: up to `k` of the types it has, and some it lacks"
    text, base, mentions, types, _ = maven_parse(doc)
    out = []
    for n, win in enumerate(windows([len(t) for t in doc["tokens"]], most)):
        sub, inside = _window_spans(text, base, doc["sentences"], win, mentions)
        found = defaultdict(list)
        for i, ss in inside.items():
            if i in types:
                for s in ss:
                    if s not in found[_human(types[i]).lower()]: found[_human(types[i]).lower()].append(s)
        if not found: continue
        asked = sample_types(list(found), type_pool, rng, absent, most=k)
        out.append({"id": f"{source}/{doc.get('id', '')}/{n}", "kind": "extraction", "source": source, "lang": "en", "text": sub,
                    "ask": {"entities": asked}, "gold": {"entities": {t: found[t] for t in asked if t in found}}} | ({"suite": suite} if suite else {}))
    return out

In [ ]:
maven = {"id": "d1", "sentences": ['The storm hit Quarto ("the town") on Monday.', "The flood that followed destroyed the bridge, and the storm ended."],
         "tokens": [["the", "storm", "hit", "quarto", "(", "``", "the", "town", "''", ")", "on", "monday", "."],
                    ["The", "flood", "that", "followed", "destroyed", "the", "bridge", ",", "and", "the", "storm", "ended", "."]],
         "events": [{"id": "EVENT_storm", "type": "Natural_disaster", "mention": [{"sent_id": 0, "offset": [1, 2]}, {"sent_id": 1, "offset": [10, 11]}]},
                    {"id": "EVENT_flood", "type": "Natural_disaster", "mention": [{"sent_id": 1, "offset": [1, 2]}]},
                    {"id": "EVENT_destroy", "type": "Destroying", "mention": [{"sent_id": 1, "offset": [4, 5]}]},
                    {"id": "EVENT_end", "type": "Process_end", "mention": [{"sent_id": 1, "offset": [11, 12]}]}],
         "TIMEX": [{"id": "TIME_mon", "type": "DATE", "sent_id": 0, "offset": [11, 12]}],
         "causal_relations": {"CAUSE": [["EVENT_storm", "EVENT_flood"], ["EVENT_flood", "EVENT_destroy"]], "PRECONDITION": []},
         "temporal_relations": {"BEFORE": [["EVENT_storm", "EVENT_end"]], "OVERLAP": [["EVENT_storm", "TIME_mon"]]},
         "subevent_relations": []}
m = maven_ere_rows(maven)
test_eq(len(m), 1)
g = m[0]["gold"]
test_eq([(h["text"], t["text"]) for h, t in g["relations"]["causes"]], [("storm", "flood"), ("flood", "destroyed")])
test_eq([(h["text"], h["start"]) for h, _ in g["relations"]["happens before"]], [("storm", 99)])      # the storm nearest "ended"
test_eq(g["relations"]["overlaps in time with"][0][1]["text"], "Monday")
test_eq(len(g["relations"]["is the same event as"]), 1)
test_eq(len(m[0]["ask"]["relations"]), 10)                                                           # every type asked: those absent are negatives
assert spans_ok(gold_example(m[0]))
test_eq(maven_ere_rows(maven, families=["causal"])[0]["ask"]["relations"], {"causes": EVENT_RELATIONS["CAUSE"][1], "is a precondition of": EVENT_RELATIONS["PRECONDITION"][1]})
few = maven_ere_rows(maven, absent=2, rng=random.Random(1))[0]                       # the types the window has, and two it lacks
test_eq(len(few["ask"]["relations"]), len(m[0]["gold"]["relations"]) + 2)
assert set(few["gold"]["relations"]) <= set(few["ask"]["relations"]) and set(m[0]["gold"]["relations"]) <= set(few["ask"]["relations"])
e = maven_event_rows(maven, ["natural disaster", "destroying", "process end", "attack", "arriving"], random.Random(0))
test_eq([s["text"] for s in e[0]["gold"]["entities"]["natural disaster"]], ["storm", "storm", "flood"])

An event's arguments, who did it, to whom, where, come from [MAVEN-Arg](https://huggingface.co/datasets/mneb/maven-arg), which annotates the same articles. They fit GLiNER2's records: a record per event type, its trigger as the first field (the one a record is anchored on) and a field per argument role. A row asks for a few of the event types its window has and one it lacks.

In [ ]:
#| export
def sentence_windows(text:str, most:int) -> list:
    "A tokenized text (tokens joined by spaces) cut into windows of whole sentences of up to `most` words: (start, end) offsets"
    ends = [m.end() for m in re.finditer(r" [.!?](?= |$)", text)]
    bounds = [0] + ends + ([len(text)] if not ends or ends[-1] < len(text) else [])
    sents = [(a + (1 if a else 0), b) for a, b in zip(bounds, bounds[1:]) if b > a + 1]
    return [(sents[w[0]][0], sents[w[-1]][1]) for w in windows([len(text[a:b].split()) for a, b in sents], most)]

def event_roles(rows) -> dict:
    "Each MAVEN-Arg event type's argument roles, as the training data uses them"
    roles = defaultdict(set)
    for r in rows:
        for t, insts in r["output"]["json_structures"].items():
            for i in insts: roles[t].update(a["role"] for a in i.get("arguments") or [])
    return {t: sorted(v) for t, v in roles.items()}

def maven_arg_rows(r:dict, roles:dict, rng:random.Random, most:int=250, per_row:int=3, absent:int=1, source:str="maven-arg",
                   suite:str|None=None) -> list:
    "Rows from MAVEN-Arg, a window of sentences each: a record per event (its trigger first, then its arguments by role) for up to `per_row` event types of the window, and `absent` types it lacks"
    text, js = r["input"], r["output"]["json_structures"]
    name = lambda t: _human(t).lower()
    out = []
    for n, (lo, hi) in enumerate(sentence_windows(text, most)):
        sub = text[lo:hi]
        inside = lambda x: lo <= x["start"] and x["end"] <= hi and text[x["start"]:x["end"]] == x["text"]
        recs = defaultdict(list)
        for t, insts in js.items():
            for i in insts:
                if not inside(i["trigger"]): continue
                rec = {"trigger": span(sub, i["trigger"]["start"] - lo, i["trigger"]["end"] - lo)}
                for a in i.get("arguments") or []:
                    if inside(a): rec.setdefault(name(a["role"]), []).append(span(sub, a["start"] - lo, a["end"] - lo))
                recs[t].append(rec)
        if not recs: continue
        present = rng.sample(sorted(recs), min(per_row, len(recs)))
        others = sorted(t for t in roles if t not in recs)
        lacking = rng.sample(others, min(absent, len(others)))
        asked = {name(t): {"trigger": f"the word or phrase that names the {name(t)} event"} | {name(x): "" for x in roles.get(t, [])} for t in present + lacking}
        out.append({"id": f"{source}/{r.get('id', '')}/{n}", "kind": "extraction", "source": source, "lang": "en", "text": sub, "ask": {"structures": asked},
                    "gold": {"structures": {name(t): recs[t] for t in present}}} | ({"suite": suite} if suite else {}))
    return out

In [ ]:
t = "the storm hit the coast . The flood destroyed the bridge and the town . Repairs began in May ."
at = lambda x: span(t, t.index(x), t.index(x) + len(x))
arg = {"input": t, "output": {"json_structures": {
    "Destroying": [{"trigger": at("destroyed"), "arguments": [{"role": "Cause", **at("The flood")}, {"role": "Patient", **at("the bridge")},
                                                              {"role": "Patient", **at("the town")}]}],
    "Process_start": [{"trigger": at("began"), "arguments": [{"role": "Time", **at("May")}]}]}}}
roles = event_roles([arg]) | {"Attack": ["Attacker", "Target"]}
w = maven_arg_rows(arg, roles, random.Random(0), most=16)
test_eq([x["text"] for x in w], ["the storm hit the coast . The flood destroyed the bridge and the town .", "Repairs began in May ."])
test_eq([s["text"] for s in w[0]["gold"]["structures"]["destroying"][0]["patient"]], ["the bridge", "the town"])
test_eq(w[1]["gold"]["structures"]["process start"][0]["time"][0], span(w[1]["text"], 17, 20))      # offsets now count from the window
test_eq(len(w[1]["ask"]["structures"]), 2)                                                        # its event type, and one it lacks
assert all(spans_ok(gold_example(x)) for x in w)
w[0]["ask"]

{'structures': {'destroying': {'trigger': 'the word or phrase that names the destroying event',
   'cause': '',
   'patient': ''},
  'process start': {'trigger': 'the word or phrase that names the process start event',
   'time': ''}}}

The [Causal News Corpus](https://huggingface.co/datasets/thagen/CNCv2) marks causes and effects in news sentences as whole clauses, `<e1>…</e1>`, with the causal pairs listed apart. Its pairs become `causes` relations between the clauses, and its sentences without any as negatives.

In [ ]:
#| export
CAUSES = {"causes": EVENT_RELATIONS["CAUSE"][1]}

def unmark(text:str) -> tuple:
    "A text without its <eN>…</eN> marks, and each mark's span in it"
    out, spans, open_, pos = [], {}, {}, 0
    for m in re.finditer(r"<(/?)(e\d+)>", text):
        out.append(text[pos:m.start()]); pos = m.end()
        at = sum(map(len, out))
        if m.group(1): spans[m.group(2)] = (open_.pop(m.group(2), at), at)
        else: open_[m.group(2)] = at
    out.append(text[pos:]); clean = "".join(out)
    return clean, {k: span(clean, a, b) for k, (a, b) in spans.items() if b > a}

def cnc_rows(rows, source:str="causal-news", suite:str|None=None) -> list:
    "Rows from the Causal News Corpus's causal pairs: the text without its marks, each pair (relationship 1: first causes second) a `causes` relation"
    out = []
    for i, r in enumerate(rows):
        text, spans = unmark(r["text"])
        rel = r["relations"]
        rel = [ast.literal_eval(x) for x in re.findall(r"\{[^}]*\}", rel)] if isinstance(rel, str) else list(rel or [])
        pairs = [[spans[x["first"]], spans[x["second"]]] for x in rel if x.get("relationship") == 1 and x["first"] in spans and x["second"] in spans]
        out.append({"id": f"{source}/{i}", "kind": "extraction", "source": source, "lang": "en", "text": text, "ask": {"relations": dict(CAUSES)},
                    "gold": {"relations": {"causes": pairs} if pairs else {}}} | ({"suite": suite} if suite else {}))
    return out

In [ ]:
cnc = [{"text": "<e1>Several thousand protesters took to the streets</e1> after <e2>six candidates were barred from the election</e2> .",
        "relations": "[{'relationship': 1, 'first': 'e2', 'second': 'e1'}]"},
       {"text": "The protest is about saving livelihoods , she said .", "relations": "[]"}]
c = cnc_rows(cnc)
test_eq([(h["text"], t["text"]) for h, t in c[0]["gold"]["relations"]["causes"]],
        [("six candidates were barred from the election", "Several thousand protesters took to the streets")])
test_eq(c[1]["gold"], {"relations": {}})
test_eq(gold_example(c[1])["absent_relations"], ["causes"])

## Statements and how they relate

Relations so far join words: an entity and another, an event's trigger and another's. Some uses need relations between whole statements. A prediction market, for one, is a question such as "Will Bitcoin close above $90,000 on June 30?", and markets depend on each other logically: one resolving yes may force another to resolve yes (it *implies* it), or make it impossible (it *rules it out*); reaching a final is a *precondition* of winning it. Finding those dependencies is how [Saguillo et al. (2025)](https://arxiv.org/abs/2508.03474) found arbitrage on Polymarket, with an LLM asked about every pair of markets.

A GLiNER2 model can read a list of statements and mark these relations between them in one pass. Three sources teach it:

- **NLI** ([MultiNLI](https://huggingface.co/datasets/nyu-mll/multi_nli)): a premise with hypotheses people wrote for it, labelled entailed, contradicted or neutral. Listed together, the premise *implies* the entailed one and *rules out* the contradicted one; the neutral one has no relation, nor do statements from different premises.
- **Cause and effect** ([e-CARE](https://huggingface.co/datasets/12ml/e-CARE)): a premise and two alternatives, one its cause (or effect). The right one *causes* the premise (or the premise causes it); the other stands unrelated.
- **Markets**, generated: questions about prices, deadlines, matches, elections and tournaments whose logical relations follow from how they are built, so their labels are exact.

`statement_list` writes statements as a list (numbered, dashed or one a line, in a random order) and marks the relations between them. A symmetric relation, *rules out*, is marked both ways. A span covers a statement without its closing punctuation.

In [ ]:
#| export
STATEMENT_RELATIONS = {"implies": "if the first is true, the second is true as well", "rules out": "the two cannot both be true",
                       "causes": EVENT_RELATIONS["CAUSE"][1], "is a precondition of": EVENT_RELATIONS["PRECONDITION"][1]}
SYMMETRIC = {"rules out"}
LIST_STYLES = ("numbered", "dashes", "lines")

def statement_list(statements:list, pairs:list, rng:random.Random, asked:list, source:str, id:str, suite:str|None=None) -> dict:
    "A row listing `statements` in a random order and style, with `pairs` (i, j, relation) between them as gold relations (a symmetric one both ways), asking the relations `asked`"
    order = list(range(len(statements))); rng.shuffle(order)
    style = rng.choice(LIST_STYLES)
    text, spans = "", {}
    for k, i in enumerate(order):
        s = statements[i].strip()
        text += ("\n" if k else "") + {"numbered": f"{k + 1}. ", "dashes": "- ", "lines": ""}[style]
        spans[i] = (len(text), len(text) + len(s.rstrip(".?!;: ")))
        text += s
    rels = defaultdict(list)
    for i, j, r in pairs:
        for a, b in ([(i, j), (j, i)] if r in SYMMETRIC else [(i, j)]):
            p = [span(text, *spans[a]), span(text, *spans[b])]
            if a != b and p not in rels[r]: rels[r].append(p)
    row = {"id": id, "kind": "extraction", "source": source, "lang": "en", "text": text,
           "ask": {"relations": {r: STATEMENT_RELATIONS[r] for r in asked}}, "gold": {"relations": dict(rels)}}
    return row | ({"suite": suite} if suite else {})

def _lists(items:list, rng:random.Random, size:tuple, words:int) -> list:
    "Items shuffled and cut into lists of `size` (low, high) items, each list kept under about `words` words"
    items = list(items); rng.shuffle(items); out = []
    while items:
        cur, n = [], 0
        for _ in range(rng.randint(*size)):
            if not items: break
            k = _words(" ".join(str(x) for x in items[-1]) if isinstance(items[-1], (list, tuple)) else str(items[-1]))
            if cur and n + k > words: break
            cur.append(items.pop()); n += k
        out.append(cur)
    return out

def nli_rows(rows, rng:random.Random, size:tuple=(1, 3), words:int=180, source:str="mnli", suite:str|None=None) -> list:
    "Lists of statements from an NLI dataset (premise, hypothesis, label: 0 entailed, 1 neutral, 2 contradicted): each premise with its hypotheses, the premise implying an entailed one and ruling out a contradicted one"
    groups = defaultdict(list)
    for r in rows:
        if r["label"] in (0, 1, 2) and r["hypothesis"].strip(): groups[r["premise"].strip()].append((r["hypothesis"].strip(), int(r["label"])))
    out = []
    for k, take in enumerate(_lists([[p] + [h for h, _ in hs] for p, hs in groups.items()], rng, size, words)):
        statements, pairs = [], []
        for p, *_ in take:
            if p in statements: continue
            statements.append(p); pi = len(statements) - 1
            for h, label in groups[p]:
                if h in statements: continue
                statements.append(h)
                if label != 1: pairs.append((pi, len(statements) - 1, "implies" if label == 0 else "rules out"))
        out.append(statement_list(statements, pairs, rng, ["implies", "rules out"], source, f"{source}/{k}", suite))
    return out

def copa_rows(rows, rng:random.Random, size:tuple=(2, 3), words:int=180, source:str="e-care", suite:str|None=None) -> list:
    "Lists of statements from cause-and-effect items (premise, question 'cause' or 'effect', choice1, choice2, label 0 or 1): the right choice causes the premise, or the premise causes it; the other choice stands unrelated"
    items = [(r["premise"], r["choice1"], r["choice2"], r["question"], int(r["label"])) for r in rows if r["question"] in ("cause", "effect")]
    out = []
    for k, take in enumerate(_lists(items, rng, size, words)):
        statements, pairs = [], []
        for p, c1, c2, q, label in take:
            b = len(statements); statements += [p, c1, c2]
            pairs.append((b + 1 + label, b, "causes") if q == "cause" else (b, b + 1 + label, "causes"))
        out.append(statement_list(statements, pairs, rng, ["causes"], source, f"{source}/{k}", suite))
    return out

In [ ]:
mnli = [{"premise": "The man is playing a guitar on stage.", "hypothesis": "A man plays an instrument.", "label": 0},
        {"premise": "The man is playing a guitar on stage.", "hypothesis": "The man is asleep.", "label": 2},
        {"premise": "The man is playing a guitar on stage.", "hypothesis": "The man is famous.", "label": 1},
        {"premise": "Prices rose sharply in May.", "hypothesis": "Prices went up.", "label": 0}]
nl = nli_rows(mnli, random.Random(0), size=(2, 2))
test_eq(len(nl), 1)
g = {k: [(h["text"], t["text"]) for h, t in v] for k, v in nl[0]["gold"]["relations"].items()}
test_eq(sorted(g["implies"]), [("Prices rose sharply in May", "Prices went up"), ("The man is playing a guitar on stage", "A man plays an instrument")])
test_eq(sorted(g["rules out"]), [("The man is asleep", "The man is playing a guitar on stage"), ("The man is playing a guitar on stage", "The man is asleep")])
print(nl[0]["text"])
care = [{"premise": "There is a light rain today.", "question": "effect", "choice1": "The soil gets wet.", "choice2": "Tourists see ripples.", "label": 0},
        {"premise": "The ice melted.", "question": "cause", "choice1": "It was cold.", "choice2": "The sun came out.", "label": 1}]
cp = copa_rows(care, random.Random(0), size=(2, 2))
test_eq(sorted((h["text"], t["text"]) for h, t in cp[0]["gold"]["relations"]["causes"]),
        [("The sun came out", "The ice melted"), ("There is a light rain today", "The soil gets wet")])
assert spans_ok(gold_example(nl[0])) and spans_ok(gold_example(cp[0]))

- The man is famous.
- The man is asleep.
- Prices went up.
- A man plays an instrument.
- The man is playing a guitar on stage.
- Prices rose sharply in May.


The market generator builds each list from a few *kinds* of market, every one with its own logic:

| kind | questions | relations |
|---|---|---|
| thresholds | will an asset close above (or below) a price on a date | above a higher price implies above a lower one; above a price rules out below it |
| deadlines | will something happen by a date | by an earlier date implies by a later one |
| matches | will a team beat another, by a margin or more | a margin implies the win, a larger margin a smaller one; either team winning rules out the other |
| elections | will a candidate, or a party's candidate, win | a candidate implies their party and rules out every other; two candidates, or two parties, rule each other out |
| stages | will a team reach a round of a tournament, or win it | a later round implies an earlier one, and the earlier is a precondition of the later; two winners rule each other out |
| reach | will an asset reach a price by a date | a higher price, or an earlier date, implies a lower price or a later date |
| counts | will something number at least (or fewer than) a count | at least more implies at least fewer; at least a count rules out fewer than it |
| ranges | will a figure fall in a range, or be at least a value | ranges rule each other out; a range implies "at least" its low end and rules out "at least" more than its high end |

A list may also join two of its questions with "and", which implies both. What a question implies or rules out follows through chains (above $100,000 implies above $90,000, which implies above $80,000), and the generator marks every relation that follows, so a list's gold is complete: two questions without a relation have none. A list also carries questions about other things, with no relation to the rest. The entities (assets, teams, candidates, events) are split in two, for training and for testing, so a test shows whether a model learned the logic or the names; the ranges kind can be held out of training whole.

In [ ]:
#| export
MARKET_WORLD = {
    "assets": [("Bitcoin", "$", [60_000, 70_000, 80_000, 90_000, 100_000, 110_000, 120_000]), ("Ethereum", "$", [2_000, 2_500, 3_000, 3_500, 4_000, 4_500]),
               ("Solana", "$", [100, 125, 150, 175, 200, 250]), ("gold", "$", [2_000, 2_200, 2_400, 2_600, 2_800, 3_000]),
               ("the S&P 500", "", [4_800, 5_000, 5_200, 5_400, 5_600, 5_800]), ("Tesla stock", "$", [150, 200, 250, 300, 350, 400]),
               ("Brent crude oil", "$", [60, 70, 80, 90, 100]), ("the Nasdaq 100", "", [16_000, 17_000, 18_000, 19_000, 20_000]),
               ("Nvidia stock", "$", [80, 100, 120, 140, 160, 180]), ("XRP", "$", [1, 2, 3, 4, 5]), ("Apple stock", "$", [180, 200, 220, 240, 260]),
               ("Dogecoin", "$", [0.1, 0.2, 0.3, 0.4, 0.5])],
    "dates": ["March 31, 2025", "June 30, 2025", "September 30, 2025", "December 31, 2025", "March 31, 2026", "June 30, 2026", "December 31, 2026"],
    "happenings": ["Apple release a foldable iPhone", "the Federal Reserve cut interest rates", "OpenAI release GPT-6", "Russia and Ukraine sign a ceasefire",
                   "the US enter a recession", "SpaceX reach orbit with Starship", "Taylor Swift announce a new album", "the UK call a general election",
                   "Elon Musk step down as CEO of Tesla", "the European Commission fine Google", "a new Pope be elected", "Netflix raise its prices in the US",
                   "the US government shut down", "Nintendo release a new console", "India land a crewed spacecraft", "Twitter rebrand back from X",
                   "the Bank of Japan raise interest rates", "Amazon announce layoffs", "Hamas release all hostages", "Germany hold a snap election"],
    "leagues": [("the Premier League", "goals", [2, 3], ["Arsenal", "Liverpool", "Manchester City", "Chelsea", "Tottenham", "Newcastle", "Aston Villa", "Manchester United"]),
                ("the NBA", "points", [5, 10, 15], ["the Lakers", "the Celtics", "the Warriors", "the Nuggets", "the Knicks", "the Bucks", "the Heat", "the Suns"]),
                ("the NFL", "points", [3, 7, 14], ["the Chiefs", "the Eagles", "the Bills", "the 49ers", "the Ravens", "the Lions", "the Cowboys", "the Packers"]),
                ("La Liga", "goals", [2, 3], ["Real Madrid", "Barcelona", "Atlético Madrid", "Sevilla", "Real Sociedad", "Villarreal", "Valencia", "Betis"])],
    "elections": [("the 2028 US presidential election", [("JD Vance", "Republican"), ("Gavin Newsom", "Democratic"), ("Marco Rubio", "Republican"),
                                                         ("Josh Shapiro", "Democratic"), ("Ron DeSantis", "Republican"), ("Pete Buttigieg", "Democratic")]),
                  ("the 2027 French presidential election", [("Jordan Bardella", "National Rally"), ("Gabriel Attal", "Renaissance"), ("Marine Le Pen", "National Rally"),
                                                             ("Édouard Philippe", "Horizons"), ("Jean-Luc Mélenchon", "La France Insoumise"), ("Raphaël Glucksmann", "Place Publique")]),
                  ("the 2026 Brazilian presidential election", [("Lula", "Workers' Party"), ("Tarcísio de Freitas", "Republicans"), ("Fernando Haddad", "Workers' Party"),
                                                                ("Michelle Bolsonaro", "Liberal Party"), ("Ratinho Júnior", "Social Democratic Party"), ("Romeu Zema", "New Party")]),
                  ("the next Canadian federal election", [("Mark Carney", "Liberal"), ("Pierre Poilievre", "Conservative"), ("Jagmeet Singh", "New Democratic"),
                                                          ("Chrystia Freeland", "Liberal"), ("Yves-François Blanchet", "Bloc Québécois"), ("Andrew Scheer", "Conservative")])],
    "tournaments": [("the Champions League", ["Real Madrid", "Barcelona", "Bayern Munich", "PSG", "Inter Milan", "Arsenal", "Liverpool", "Manchester City"],
                     ["reach the quarterfinals of", "reach the semifinals of", "reach the final of", "win"]),
                    ("the World Cup", ["Brazil", "France", "Argentina", "England", "Spain", "Germany", "Portugal", "the Netherlands"],
                     ["reach the round of 16 at", "reach the quarterfinals at", "reach the semifinals at", "reach the final at", "win"]),
                    ("Wimbledon", ["Carlos Alcaraz", "Jannik Sinner", "Novak Djokovic", "Daniil Medvedev", "Alexander Zverev", "Taylor Fritz", "Jack Draper", "Holger Rune"],
                     ["reach the quarterfinals of", "reach the semifinals of", "reach the final of", "win"]),
                    ("the NBA playoffs", ["the Celtics", "the Thunder", "the Nuggets", "the Knicks", "the Cavaliers", "the Lakers", "the Timberwolves", "the Pacers"],
                     ["reach the second round of", "reach the conference finals in", "reach the NBA Finals through", "win the championship in"])],
    "counts": [("Tesla deliver", "vehicles in Q3 2025", [350_000, 400_000, 450_000, 500_000]), ("Apple sell", "iPhones in 2025", [200_000_000, 220_000_000, 240_000_000]),
               ("the US economy add", "jobs in May", [100_000, 150_000, 200_000, 250_000]), ("Netflix add", "subscribers in Q2", [5_000_000, 7_500_000, 10_000_000]),
               ("SpaceX launch", "rockets in 2025", [100, 125, 150, 175]), ("Lionel Messi score", "goals this season", [10, 15, 20, 25]),
               ("Boeing deliver", "planes in 2025", [300, 350, 400, 450]), ("Shohei Ohtani hit", "home runs this season", [40, 45, 50, 55]),
               ("Spotify add", "paying subscribers in Q3", [5_000_000, 8_000_000, 10_000_000]), ("Toyota sell", "cars in 2025", [10_000_000, 10_500_000, 11_000_000]),
               ("Erling Haaland score", "Premier League goals this season", [20, 25, 30, 35]), ("Taylor Swift's next album sell", "copies in its first week", [1_000_000, 1_500_000, 2_000_000]),
               ("Microsoft lay off", "employees in 2025", [5_000, 10_000, 15_000]), ("Airbus deliver", "planes in 2025", [700, 750, 800, 850]),
               ("Meta add", "daily users in Q3", [50_000_000, 75_000_000, 100_000_000]), ("Nvidia ship", "data-center GPUs in Q3", [1_000_000, 1_500_000, 2_000_000])],
    "ranges": [("US CPI inflation for May", "%", 2.0, 0.5, 5, 0.1), ("the number of named Atlantic storms in 2025", "", 10, 5, 4, 1),
               ("the US unemployment rate for June", "%", 3.5, 0.5, 4, 0.1), ("the number of Elon Musk's posts on X this week", "", 100, 50, 5, 1),
               ("Germany's GDP growth for 2025", "%", -1.0, 0.5, 5, 0.1), ("the number of US measles cases in 2025", "", 500, 250, 4, 1)]}
MARKET_KINDS = ("thresholds", "deadlines", "matches", "elections", "stages", "reach", "counts", "ranges")     # the last held out of training by default

def _money(x, unit:str) -> str:
    v = f"{x:,}" if float(x).is_integer() else f"{x:,.2f}".rstrip("0").rstrip(".")
    return f"{unit}{v}" if unit != "%" else f"{x:.1f}%"

MARKET_SPLIT = 171                 # the salt of the name hash: with it, every pool keeps enough names on both sides

def _part(xs:list, part:str) -> list:
    "The training or the test names of a pool: a name's side follows a hash of it, so a name shared by two pools (a club in a league and a cup) is on the same side of both"
    side = lambda x: zlib.crc32(f"{MARKET_SPLIT}:{x[0] if isinstance(x, tuple) else x}".encode()) % 2 == 0
    return [x for x in xs if side(x) == (part == "train")]

def _market(kind:str, rng:random.Random, w:dict, part:str, used:set):
    "One market group of `kind`: its questions as clauses ('Bitcoin close above $90,000 on June 30, 2025'), and the relations that hold between them; None when its entity is taken"
    if kind == "thresholds":
        (a, unit, levels), d = rng.choice(_part(w["assets"], part)), rng.choice(w["dates"])
        if (a, d) in used or ("reach", a) in used: return None
        used |= {(a, d), ("asset", a)}; xs = sorted(rng.sample(levels, rng.randint(2, min(4, len(levels)))))
        qs = [(x, s) for x in xs for s in ("above", "below") if rng.random() < 0.6] or [(xs[0], "above"), (xs[-1], "above")]
        verb = rng.choice(["close", "be", "trade"])
        clauses = [f"{a} {verb} {s} {_money(x, unit)} on {d}" for x, s in qs]
        pairs = [(i, j, "implies") for i, (x, s) in enumerate(qs) for j, (y, t) in enumerate(qs) if s == t and i != j and (x > y if s == "above" else x < y)]
        pairs += [(i, j, "rules out") for i, (x, s) in enumerate(qs) for j, (y, t) in enumerate(qs) if s == "above" and t == "below" and y <= x]
        return clauses, pairs
    if kind == "deadlines":
        h = rng.choice(_part(w["happenings"], part))
        if h in used: return None
        used.add(h); ds = sorted(rng.sample(range(len(w["dates"])), rng.randint(2, 3)))
        return [f"{h} by {w['dates'][i]}" for i in ds], [(i, j, "implies") for i in range(len(ds)) for j in range(i + 1, len(ds))]
    if kind == "matches":
        name, unit, margins, teams = rng.choice(w["leagues"])
        t1, t2 = rng.sample(_part(teams, part), 2)
        if frozenset((t1, t2)) in used: return None
        used.add(frozenset((t1, t2))); ks = sorted(rng.sample(margins, rng.randint(0, min(2, len(margins)))))
        clauses = [f"{t1} beat {t2} in {name}"] + [f"{t1} beat {t2} by {k} or more {unit} in {name}" for k in ks]
        pairs = [(i, 0, "implies") for i in range(1, len(clauses))] + [(j, i, "implies") for i in range(1, len(clauses)) for j in range(i + 1, len(clauses))]
        if rng.random() < 0.6: clauses.append(f"{t2} beat {t1} in {name}"); pairs.append((0, len(clauses) - 1, "rules out"))
        return clauses, pairs
    if kind == "elections":
        e, people = rng.choice(w["elections"])
        if e in used: return None
        pool = _part(people, part); used.add(e); cs = rng.sample(pool, rng.randint(2, min(3, len(pool))))
        parties = _unique(p for _, p in cs); shown = [p for p in parties if rng.random() < 0.6]
        clauses = [f"{c} win {e}" for c, _ in cs] + [f"the {p} candidate win {e}" for p in shown]
        n = len(cs)
        pairs = [(i, j, "rules out") for i in range(n) for j in range(i + 1, n)]
        pairs += [(i, n + shown.index(p), "implies") for i, (_, p) in enumerate(cs) if p in shown]
        pairs += [(i, n + k, "rules out") for i, (_, p) in enumerate(cs) for k, q in enumerate(shown) if q != p]   # and no other party wins
        pairs += [(n + i, n + j, "rules out") for i in range(len(shown)) for j in range(i + 1, len(shown))]
        return clauses, pairs
    if kind == "stages":
        name, teams, stages = rng.choice(w["tournaments"])
        if name in used: return None
        used.add(name); t = rng.choice(_part(teams, part)); ss = sorted(rng.sample(range(len(stages)), rng.randint(2, min(3, len(stages)))))
        clauses = [f"{t} {stages[s]} {name}" for s in ss]
        pairs = [(j, i, "implies") for i in range(len(ss)) for j in range(i + 1, len(ss))]
        pairs += [(i, j, "is a precondition of") for i in range(len(ss)) for j in range(i + 1, len(ss))]
        if ss[-1] == len(stages) - 1 and rng.random() < 0.5:
            other = rng.choice([x for x in _part(teams, part) if x != t])
            clauses.append(f"{other} {stages[-1]} {name}"); pairs.append((len(ss) - 1, len(clauses) - 1, "rules out"))
        return clauses, pairs
    if kind == "reach":                                 # reaching a price by a date: a higher price, or an earlier date, implies the rest
        a, unit, levels = rng.choice(_part(w["assets"], part))
        if ("asset", a) in used or ("reach", a) in used: return None
        used |= {("asset", a), ("reach", a)}
        xs, ds = sorted(rng.sample(levels, rng.randint(1, min(3, len(levels))))), sorted(rng.sample(range(len(w["dates"])), rng.randint(1, 3)))
        qs = rng.sample([(x, d) for x in xs for d in ds], min(len(xs) * len(ds), rng.randint(2, 4)))
        if len(qs) < 2: return None
        clauses = [f"{a} reach {_money(x, unit)} by {w['dates'][d]}" for x, d in qs]
        return clauses, [(i, j, "implies") for i, (x, d) in enumerate(qs) for j, (y, e) in enumerate(qs) if i != j and y <= x and e >= d]
    if kind == "counts":
        subj, what, levels = rng.choice(_part(w["counts"], part))
        if subj in used: return None
        used.add(subj); ns = sorted(rng.sample(levels, rng.randint(2, min(3, len(levels)))))
        qs = [(x, s) for x in ns for s in ("at least", "fewer than") if rng.random() < 0.6] or [(ns[0], "at least"), (ns[-1], "at least")]
        clauses = [f"{subj} {s} {x:,} {what}" for x, s in qs]
        pairs = [(i, j, "implies") for i, (x, s) in enumerate(qs) for j, (y, t) in enumerate(qs) if s == t and i != j and (x > y if s == "at least" else x < y)]
        pairs += [(i, j, "rules out") for i, (x, s) in enumerate(qs) for j, (y, t) in enumerate(qs) if s == "at least" and t == "fewer than" and y <= x]
        return clauses, pairs
    if kind == "ranges":
        q, unit, lo, step, k, res = rng.choice(_part(w["ranges"], part))
        if q in used: return None
        used.add(q)
        fmt = lambda x: f"{x:.1f}%" if unit == "%" else f"{round(x):,}"
        bounds = [round(lo + i * step, 6) for i in range(k)]
        bs = sorted(rng.sample(range(k - 1), rng.randint(2, min(3, k - 1))))
        clauses = [f"{q} be between {fmt(bounds[i])} and {fmt(bounds[i + 1] - res)}" for i in bs]
        ats = sorted(rng.sample(range(1, k), rng.randint(1, 2)))
        clauses += [f"{q} be at least {fmt(bounds[i])}" for i in ats]
        n = len(bs)
        pairs = [(i, j, "rules out") for i in range(n) for j in range(i + 1, n)]
        pairs += [(i, n + a, "implies") for i, b in enumerate(bs) for a, at in enumerate(ats) if bounds[at] <= bounds[b]]
        pairs += [(i, n + a, "rules out") for i, b in enumerate(bs) for a, at in enumerate(ats) if bounds[at] >= bounds[b + 1]]
        pairs += [(n + a, n + c, "implies") for a, x in enumerate(ats) for c, y in enumerate(ats) if x > y]
        return clauses, pairs
    raise ValueError(f"no market kind {kind!r}")

def market_closure(n:int, pairs:list) -> list:
    "Every relation that follows from `pairs` among `n` questions: implication and precondition through chains, and what implies one of two exclusive questions excluding the other"
    imp = {(a, b) for a, b, r in pairs if r == "implies"}
    pre = {(a, b) for a, b, r in pairs if r == "is a precondition of"}
    for rel in (imp, pre):
        for k in range(n):
            for a in range(n):
                if (a, k) in rel: rel |= {(a, b) for b in range(n) if (k, b) in rel}
    above = lambda x: {x} | {a for a, b in imp if b == x}
    excl = {frozenset((x, y)) for a, b, r in pairs if r == "rules out" for x in above(a) for y in above(b) if x != y}
    return ([(a, b, "implies") for a, b in sorted(imp) if a != b] + [(a, b, "is a precondition of") for a, b in sorted(pre) if a != b]
            + [(*sorted(e), "rules out") for e in sorted(excl, key=sorted)])

def market_rows(n:int, seed:int=0, part:str="train", kinds:tuple=MARKET_KINDS[:-1], world:dict=MARKET_WORLD, source:str="markets",
                suite:str|None=None, fillers:tuple|None=None) -> list:
    "`n` lists of market questions, each from one or two market groups of `kinds` (sometimes two questions joined by 'and') and up to three unrelated questions, every relation that holds marked; `part` picks the training or the test half of the entities, `fillers` the kinds the unrelated questions come from (default: every kind but the last)"
    rng, out = random.Random(f"{seed}:{part}"), []
    while len(out) < n:
        used, clauses, pairs = set(), [], []
        for kind in rng.sample(list(kinds), min(len(kinds), rng.randint(1, 2))):
            if (got := _market(kind, rng, world, part, used)) is None: continue
            b = len(clauses); clauses += got[0]; pairs += [(i + b, j + b, r) for i, j, r in got[1]]
        if not clauses: continue
        for _ in range(rng.randint(0, 3)):
            got = _market(rng.choice(fillers or MARKET_KINDS[:-1]), rng, world, part, used)
            if got: clauses.append(rng.choice(got[0]))
        if len(clauses) >= 2 and rng.random() < 0.3:
            i, j = rng.sample(range(len(clauses)), 2)
            clauses.append(f"{clauses[i]} and {clauses[j]}"); pairs += [(len(clauses) - 1, i, "implies"), (len(clauses) - 1, j, "implies")]
        if len(clauses) > 10: continue
        qs = [f"Will {c}?" for c in clauses]
        asked = ["implies", "rules out", "is a precondition of"]
        out.append(statement_list(qs, market_closure(len(qs), pairs), rng, asked, source, f"{source}-{part}/{len(out)}", suite))
    return out

In [ ]:
test_eq(sorted(market_closure(3, [(0, 1, "implies"), (1, 2, "implies")])), [(0, 1, "implies"), (0, 2, "implies"), (1, 2, "implies")])
reach = market_rows(300, seed=4, kinds=("reach",))
date = {d: i for i, d in enumerate(MARKET_WORLD["dates"])}
parse = lambda q: (lambda m: (float(m[2].replace(",", "")), date[m[3]]) if m else None)(re.match(r"Will (.+) reach \$?([\d.,]+) by (.+?)\??$", q))
checked = 0
for rr in reach:                                     # each implication between two reach questions: a price no higher, a date no earlier
    for hh, tt in rr["gold"]["relations"].get("implies", []):
        if " and " in hh["text"] + tt["text"] or not (parse(hh["text"]) and parse(tt["text"])): continue  # a joined question holds two
        (x1, d1), (x2, d2) = parse(hh["text"]), parse(tt["text"])
        assert x2 <= x1 and d2 >= d1, (hh["text"], tt["text"]); checked += 1
assert checked > 100
test_eq(market_closure(3, [(0, 1, "implies"), (1, 2, "rules out")]), [(0, 1, "implies"), (0, 2, "rules out"), (1, 2, "rules out")])
mk = market_rows(200, seed=0)
assert all(spans_ok(gold_example(r)) for r in mk)
test_eq({len(r["text"].splitlines()) <= 10 for r in mk}, {True})
pools = lambda w: [x[0] if isinstance(x, tuple) else x for x in w["assets"] + w["happenings"] + w["ranges"] + w["counts"]] + \
                  [t for l in w["leagues"] for t in l[3]] + [c for _, ps in w["elections"] for c, _ in ps] + [t for _, ts, _ in w["tournaments"] for t in ts]
test_names = {x for x in pools(MARKET_WORLD) if _part([x], "test")}
train_text = " ".join(r["text"] for r in market_rows(300, part="train", kinds=MARKET_KINDS))
for _, ps in MARKET_WORLD["elections"]:                        # party names may hold a place name ("La France Insoumise")
    for _, party in ps: train_text = train_text.replace(party, "")
assert not [x for x in test_names if re.search(rf"(?<![\w]){re.escape(x)}(?![\w])", train_text)]     # no test name in training lists
el, party, checked = market_rows(300, seed=5, kinds=("elections",)), {c: p for _, ps in MARKET_WORLD["elections"] for c, p in ps}, Counter()
for er in el:                                        # a candidate's win implies their party's and rules out every other party's
    eqs = [re.sub(r"^(\d+\. |- )", "", x).rstrip("?") for x in er["text"].splitlines()]
    eg = {(h["text"], t["text"], k) for k, v in er["gold"]["relations"].items() for h, t in v}
    for x in eqs:
        for y in eqs:
            mx, my = re.match(r"Will (.+?) win (the .+)$", x), re.match(r"Will the (.+) candidate win (the .+)$", y)
            if " and " in x + y or not (mx and my and mx[1] in party and mx[2] == my[2]): continue
            want = "implies" if party[mx[1]] == my[1] else "rules out"
            assert (x, y, want) in eg, (x, want, y); checked[want] += 1
assert checked["implies"] and checked["rules out"]
r = next(r for r in mk if "is a precondition of" in r["gold"]["relations"])
print(r["text"]); {k: [(h["text"], t["text"]) for h, t in v] for k, v in r["gold"]["relations"].items()}

Will XRP reach $5 by June 30, 2025?
Will XRP reach $1 by June 30, 2025?
Will Portugal win the World Cup?
Will XRP reach $1 by June 30, 2025 and England win the World Cup?
Will Portugal reach the semifinals at the World Cup?
Will the New Democratic candidate win the next Canadian federal election?
Will England win the World Cup?
Will Dogecoin close below $0.3 on December 31, 2025?


{'implies': [('Will XRP reach $5 by June 30, 2025',
   'Will XRP reach $1 by June 30, 2025'),
  ('Will Portugal win the World Cup',
   'Will Portugal reach the semifinals at the World Cup'),
  ('Will XRP reach $1 by June 30, 2025 and England win the World Cup',
   'Will XRP reach $1 by June 30, 2025'),
  ('Will XRP reach $1 by June 30, 2025 and England win the World Cup',
   'Will England win the World Cup')],
 'is a precondition of': [('Will Portugal reach the semifinals at the World Cup',
   'Will Portugal win the World Cup')],
 'rules out': [('Will Portugal win the World Cup',
   'Will England win the World Cup'),
  ('Will England win the World Cup', 'Will Portugal win the World Cup'),
  ('Will Portugal win the World Cup',
   'Will XRP reach $1 by June 30, 2025 and England win the World Cup'),
  ('Will XRP reach $1 by June 30, 2025 and England win the World Cup',
   'Will Portugal win the World Cup')]}

In [ ]:
rg = market_rows(100, seed=1, part="test", kinds=("ranges",))
between, least = re.compile(r"^Will (.+) be between (-?[\d.,]+)%? and (-?[\d.,]+)%?$"), re.compile(r"^Will (.+) be at least (-?[\d.,]+)%?$")
num = lambda x: float(x.replace(",", ""))
checked = Counter()
for r in rg:
    for rel, ps in r["gold"]["relations"].items():
        for h, t in ps:
            b, a = between.match(h["text"]), least.match(t["text"])
            if not (b and a and b[1] == a[1]): continue
            assert (num(a[2]) <= num(b[2])) if rel == "implies" else (num(a[2]) > num(b[3])), (rel, h["text"], t["text"])
            checked[rel] += 1
assert checked["implies"] and checked["rules out"]                   # a range implies "at least" its low end and rules out "at least" past its high end
print(next(r for r in rg if "between" in r["text"])["text"])

Will the number of named Atlantic storms in 2025 be at least 20?
Will the number of named Atlantic storms in 2025 be between 15 and 19?
Will the number of named Atlantic storms in 2025 be between 20 and 24?
Will Brent crude oil close above $100 on March 31, 2025?
Will SpaceX launch fewer than 175 rockets in 2025?
Will the number of named Atlantic storms in 2025 be at least 15?


## Mixing

A training mix takes rows from many sources. `cap` keeps at most so many from each, so a source with a million rows does not drown one with three thousand; `fits` keeps rows short enough to train on whole (GLiNER2's encoder reads at most 512 tokens; longer texts would be cut, and gold spans past the cut would turn into false negatives). It counts words as GLiNER2 splits them, punctuation apart: a log line such as `time=09:11:11 user=lucas` is two words to a space but ten to GLiNER2.

In [ ]:
#| export
def cap(rows:list, most:int, key:str="source", seed:int=0) -> list:
    "At most `most` rows of each `key` value, drawn at random, kept in their order"
    rng, groups = random.Random(seed), defaultdict(list)
    for i, r in enumerate(rows): groups[r.get(key)].append(i)
    keep = sorted(i for ix in groups.values() for i in (ix if len(ix) <= most else rng.sample(ix, most)))
    return [rows[i] for i in keep]

def gliner_words(text:str) -> int:
    "How many words GLiNER2's default splitter reads in `text` (a punctuation mark is a word of its own)"
    return sum(1 for _ in _splitter()(text, lower=False))

def fits(row:dict, words:int=300, tokens:int=700) -> bool:
    "Whether a row is short enough to train on whole: its text up to `words` words as GLiNER2 reads them, its text and ask up to `tokens` tokens"
    return gliner_words(row["text"]) <= words and row_length(row) <= tokens

In [ ]:
many = [{"source": "big", "text": "x"} for _ in range(50)] + [{"source": "small", "text": "y"} for _ in range(3)]
test_eq(Counter(r["source"] for r in cap(many, 10)), {"big": 10, "small": 3})
assert fits(rows[0]) and not fits(rows[0] | {"text": "word " * 400})
test_eq(gliner_words("time=09:11:11 user=lucas"), 10)

## Training on gold

`GoldCollator` builds GLiNER2's batches from training records. In training it lets GLiNER2's processor do what it does when Fastino trains these models: shuffle the order of what is asked, sometimes drop labels or record fields, sometimes list a relation's tail before its head, sometimes rename a decision's labels to placeholders ("label 1", "label 2") with the real names as their descriptions, so the model reads the schema instead of memorising it. That sampling draws from Python's random numbers, so each batch seeds them from the run's seed, its epoch and its records: a run paused and resumed sees the same batches.

GLiNER2's defaults are strong: half of the label sets renamed, up to half of the labels dropped, a fifth of the gold relations dropped (which trains them as absent). `SAMPLING`, the learner's default, keeps every kind of sampling but milder, and drops no gold relation.

A decision's soft targets replace GLiNER2's 0/1 targets when its labels kept their names; a renamed one keeps the hard target.

In [ ]:
#| export
def _record(e): return json.loads(e) if isinstance(e, str) else e

SAMPLING = {"synthetic_label_prob": 0.2, "remove_classification_label_prob": 0.2, "include_true_label_prob": 0.8, "remove_relations_prob": 0.0,
            "remove_json_structure_prob": 0.1, "remove_json_field_prob": 0.1, "synthetic_entity_label_prob": 0.1, "swap_head_tail_prob": 0.2}

def set_sampling(processor, sampling:dict|None) -> dict:
    "Change the training-time sampling of a GLiNER2 processor (fields of its `SamplingConfig`); returns the settings now in force"
    cfg = copy.copy(processor.sampling_config)
    for k, v in (sampling or {}).items():
        if not hasattr(cfg, k): raise ValueError(f"GLiNER2's SamplingConfig has no {k!r}")
        setattr(cfg, k, v)
    processor.sampling_config = cfg
    return dict(vars(cfg)) if hasattr(cfg, "__dict__") else {k: getattr(cfg, k) for k in cfg.__dataclass_fields__}

class GoldCollator:
    "GLiNER2 batches from gold training records: GLiNER2's own training-time sampling seeded per batch (or none, for validation), absent relations as negatives, soft decision targets where the labels kept their names; a record the processor refuses is dropped (`drop`) or raises"
    def __init__(self, processor, train:bool=True, max_len:int|None=None, max_gold:int=64, drop:bool=True):
        self.processor, self.train, self.max_len, self.max_gold, self.drop = processor, train, max_len, max_gold, drop
    def _collate(self, examples):
        kw = {"build_targets": True} if "build_targets" in inspect.signature(self.processor._collate_batch).parameters else {}
        return self.processor._collate_batch([(e["input"], json.loads(json.dumps(e["output"]))) for e in examples], max_len=self.max_len,
                                             error_policy="raise", **kw)
    def __call__(self, items:list):
        self.processor.is_training = self.train
        examples = [_record(e) for e, _ in items]
        if self.train: random.seed(items[0][1])
        try: return self._batch(examples)
        except Exception:                                   # a record the processor refuses, at any step: drop it, keep the rest of the batch
            if not self.drop: raise
            ok = []
            for e in examples:
                try: self._batch([e]); ok.append(e)
                except Exception: pass
            if not ok: raise
            if self.train: random.seed(items[0][1])
            return self._batch(ok)
    def _batch(self, examples:list):
        p, batch = self.processor, self._collate(examples)
        for i, e in enumerate(examples):
            absent, soft = set(e.get("absent_relations") or ()), e.get("soft") or {}
            for j, (kind, toks) in enumerate(zip(batch.task_types[i], batch.schema_tokens_list[i])):
                if kind == "relations" and toks[2] in absent: batch.structure_labels[i][j] = [0, []]
                elif kind == "classifications" and soft:
                    probs = soft.get(toks[2]) or next((v for t, v in soft.items() if toks[2].startswith(t + ":")), None)
                    labels = _labels_of(toks)
                    if probs and labels and all(l in probs for l in labels): batch.structure_labels[i][j] = [probs[l] for l in labels]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return p._add_boundary_metadata(batch, "boundary", is_training=self.train, build_targets=True, max_gold_per_query=self.max_gold,
                                            on_capacity_exceeded="truncate_with_warning")

In [ ]:
tiny = load_extractor(FIX / "base")
col = GoldCollator(tiny.processor, train=False)
b = col([(gold_example(row), 0)])
kinds = list(b.task_types[0])
test_eq(b.structure_labels[0][kinds.index("classifications")], [0.3, 0.7])             # soft targets, in the order the labels were written
test_eq(b.structure_labels[0][[i for i, t in enumerate(b.schema_tokens_list[0]) if t[2] == "happens before"][0]], [0, []])
seeded = [GoldCollator(tiny.processor, train=True)([(gold_example(row), 7)]).schema_tokens_list[0] for _ in range(2)]
test_eq(seeded[0], seeded[1])                                                           # the same seed, the same sampling
broken = gold_example(row) | {"output": {"entities": {"event": [{"text": "storm", "start": 4, "end": 400}]}}}
test_eq(len(col([(gold_example(row), 0), (broken, 0)]).task_types), 1)                 # the refused record is dropped ...
test_fail(lambda: GoldCollator(tiny.processor, train=False, drop=False)([(broken, 0)])) # ... or raises
long = gold_example(row | {"text": "word " * 20 + row["text"], "gold": {"entities": {"event": [span("word " * 20 + row["text"], 104, 109)]}}})
test_eq(len(GoldCollator(tiny.processor, train=False, max_len=20)([(gold_example(row), 0), (long, 0)]).task_types), 1)   # a gold span past the cut

`GlinerLearner` trains on gold records. It is the distillation notebook's `DistillLearner` (checkpoints, a deadline, pause and resume) with what gold training needs:

- **A full fine-tune or an adapter.** Given a plain GLiNER2 model it trains every weight, the encoder at `lr` and the heads at `head_lr`, as GLiNER2's own trainer does (1e-5 and 5e-4 by default there). Given an adapter (`adapter_model`), the adapter's matrices train at `lora_lr` and the heads it carries whole at `head_lr`.
- **Losses by task.** GLiNER2's forward pass gives one loss per head: the decisions (`classification_loss`), the relations (`relation_loss`, computed in training mode only), and the spans of entities and records (the rest). The step minimises their weighted sum; with `balance=True` each is first divided by its own running average, so that the span losses, which run in the tens and hundreds, do not drown the decisions, which run below one.
- **Records as JSON lines.** A mix of a few hundred thousand records takes much less memory as strings, which the collator parses a batch at a time; their lengths are measured once.

In [ ]:
#| export
def adapter_model(model, r:int=64, alpha:float|None=None, heads:bool=True):
    "A LoRA adapter over a GLiNER2 model: rank `r` on the encoder's linear layers (their bias changes inside the adapter, so switching it off gives the model back), the heads trained whole"
    from peft import LoraConfig, get_peft_model
    keep = [n for n, _ in model.named_children() if n != "encoder"] if heads else None
    return get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha or r, target_modules=lora_targets(model), lora_dropout=0.0, bias="none",
                                            lora_bias=True, modules_to_save=keep))

class GlinerLearner(DistillLearner):
    "Train a GLiNER2 model on gold records, whole or as an adapter: GLiNER2's training-time sampling, soft decision targets, a loss per task"
    def __init__(self, model, train:list, valid:list|None=None, bs:int=16, lr:float=1e-5, head_lr:float=5e-4, lora_lr:float=2e-4,
                 weight_decay:float=0.01, weights:dict|None=None, balance:bool=False, sampling:dict|None=SAMPLING, max_len:int|None=384, max_tokens:int|None=4096,
                 warmup:float=0.06, seed:int=0, precision:str="auto", workers:int=0, log_every:int=25, verbose:bool=True, device=None):
        super().__init__(model, [], [], bs=bs, lr=lora_lr, full_lr=head_lr, max_len=max_len, max_tokens=max_tokens, warmup=warmup, seed=seed,
                         precision=precision, workers=workers, log_every=log_every, verbose=verbose, device=device)
        self.train, self.valid = list(train), list(valid or [])
        if not hasattr(model, "peft_config"):
            for p in model.parameters(): p.requires_grad_(True)
        self.enc_lr, self.head_lr, self.lora_lr, self.weight_decay, self.balance = lr, head_lr, lora_lr, weight_decay, balance
        self.weights = {"decision": 1.0, "span": 1.0, "relation": 1.0} | dict(weights or {})
        self.lengths = [_length(e) for e in self.train]; self.valid_lengths = [_length(e) for e in self.valid]
        self.kinds = Counter(_record(e).get("kind") for e in self.train) if len(self.train) < 50_000 else None
        m = _unwrap(model); g = m.boundary_settings.max_gold_per_query
        self.sampling = set_sampling(m.processor, sampling)
        self.collate, self.valid_collate = GoldCollator(m.processor, True, max_len, g), GoldCollator(m.processor, False, max_len, g)
        self.state["scales"] = {}
    def __repr__(self):
        return f"GlinerLearner({len(self.train):,} train · {len(self.valid):,} valid · {'adapter' if hasattr(self.model, 'peft_config') else 'full'} · bs {self.bs} · {self.precision} · {self.device})"

def _length(e) -> int:
    e = _record(e)
    return e.get("length") or row_length({"text": e["input"], "ask": {}}) + 50

In [ ]:
#| export
def _group(name:str) -> str:
    "Which learning rate a parameter takes: an adapter's matrices, the encoder, or the heads"
    return "lora" if "lora_" in name else "encoder" if name.startswith("encoder.") or ".encoder." in name else "heads"

@patch
def _setup(self:GlinerLearner, epochs:int):
    "AdamW over three groups (the adapter's matrices, the encoder, the heads), decay on weight matrices only; warmup, then a cosine decay to zero"
    self.model.to(self.device)
    lrs = {"lora": self.lora_lr, "encoder": self.enc_lr, "heads": self.head_lr}
    groups = defaultdict(list)
    for n, p in self.trainable(): groups[(_group(n), p.ndim >= 2 and "embeddings" not in n)].append(p)
    self.opt = torch.optim.AdamW([{"params": ps, "lr": lrs[g], "weight_decay": self.weight_decay if decay else 0.0} for (g, decay), ps in groups.items()])
    self.batches = [epoch_batches(self.train, self.bs, self.seed, e, max_tokens=self.max_tokens, lengths=self.lengths) for e in range(epochs)]
    total = sum(len(b) for b in self.batches); warm = max(1, int(self.warmup * total))
    f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    self.sched = torch.optim.lr_scheduler.LambdaLR(self.opt, f)
    self.state["epochs"], self.state["max_steps"] = epochs, total

@patch
def _loader(self:GlinerLearner, examples:list, batches:list, collate=None):
    "A DataLoader over `batches` of `examples`, each batch carrying its seed (the run's seed, the epoch and the batch's first records)"
    ep = self.state["epoch"]
    items = [(examples[i], zlib.crc32(f"{self.seed}:{ep}:{':'.join(map(str, b[:4]))}".encode())) for b in batches for i in b]
    return DataLoader(items, batch_sampler=_Offsets(batches), collate_fn=collate or self.collate, num_workers=self.workers,
                      generator=torch.Generator().manual_seed(self.seed))

@patch
def _parts(self:GlinerLearner, batch) -> dict:
    "The step's losses by head: decisions, relations, and the spans of entities and records (the rest of GLiNER2's loss)"
    batch = batch.to(self.device)
    with torch.autocast(self.device.type, dtype=torch.bfloat16, enabled=self.precision == "bf16"):
        out = self.model(batch)
    L, zero = out.losses or {}, torch.zeros((), device=self.device)
    dec = L["classification_loss"].float() if L.get("classification_loss") is not None else zero
    rel = L["relation_loss"].float() if L.get("relation_loss") is not None else zero
    return {"decision": dec, "relation": rel, "span": out.loss.float() - dec - rel if out.loss is not None else zero}

@patch
def _combine(self:GlinerLearner, parts:dict) -> torch.Tensor:
    "The weighted sum of the losses; with `balance`, each first divided by its running average (kept with the checkpoint)"
    if not self.balance: return sum(self.weights.get(k, 1.0) * v for k, v in parts.items())
    sc, total = self.state["scales"], 0.0
    for k, v in parts.items():
        x = float(v.detach())
        if x > 0 and self.model.training: sc[k] = 0.98 * sc[k] + 0.02 * x if k in sc else x
        total = total + self.weights.get(k, 1.0) * v / max(sc.get(k, 1.0), 1e-6)
    return total

@patch
def validate(self:GlinerLearner) -> dict:
    "Mean losses by head over the validation records, without sampling or dropout; a head's mean over the batches that train it"
    if not self.valid: return {}
    m = _unwrap(self.model)
    rng = (random.getstate(), torch.get_rng_state(), torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    self.model.eval(); m.training = True              # the top module only: GLiNER2 computes the relation loss, dropout and sampling stay off
    sums, counts = defaultdict(float), defaultdict(int)
    try:
        with torch.no_grad():
            for batch in self._loader(self.valid, epoch_batches(self.valid, self.bs, self.seed, 0, max_tokens=self.max_tokens, lengths=self.valid_lengths),
                                      self.valid_collate):
                for name, v in self._parts(batch).items():
                    if float(v) != 0: sums[name] += float(v); counts[name] += 1
    finally:
        m.training = False; self.model.train()
        random.setstate(rng[0]); torch.set_rng_state(rng[1])
        if rng[2] is not None: torch.cuda.set_rng_state_all(rng[2])
    return {f"valid_{k}": sums[k] / counts[k] for k in sums}

@patch
def _load(self:GlinerLearner, path):
    "Continue from a checkpoint, the losses' running averages included"
    DistillLearner._load(self, path)
    self.state["scales"] = json.loads((Path(path) / "sysone_state.json").read_text()).get("scales") or {}

Training the tiny model, whole, on a handful of gold rows: the losses of every head fall, and the decision's probabilities move to the soft targets.

In [ ]:
def gold_rows():
    rs = [row] + rows + [jev_extraction_row(ner), jev_extraction_row(qa), r] + d + m + c[:1] + w
    return [json.dumps(gold_example(x)) for x in rs]
train = gold_rows()
def tiny_gold(**kw):
    torch.manual_seed(0)
    return GlinerLearner(load_extractor(FIX / "base"), train, train[:4], bs=4, lr=1e-3, head_lr=1e-2, warmup=0.0, verbose=False, max_tokens=None, **kw)
learn = tiny_gold()
first = learn.validate()
log = learn.fit(6)
last = log[-1]
print({k: round(v, 3) for k, v in first.items()}, "→", {k: v for k, v in last.items() if k.startswith("valid")})
assert last["valid_decision"] < first["valid_decision"] and last["valid_span"] < first["valid_span"] and last["valid_relation"] < first["valid_relation"]

{'valid_decision': 0.682, 'valid_relation': 0.76, 'valid_span': 3.728} → {'valid_decision': 0.675, 'valid_relation': 0.0305, 'valid_span': 1.9228}


A paused run resumes where it stopped and ends with the weights an uninterrupted run gets, the sampling included:

In [ ]:
ck = Path(tempfile.mkdtemp()) / "learner-1"
whole = tiny_gold(balance=True); whole.fit(2)
first_half = tiny_gold(balance=True).checkpoints(ck, every=None)
n = {"steps": 0}; orig = first_half._parts
def counting(batch):
    if torch.is_grad_enabled():                                # training steps, not validation
        n["steps"] += 1
        if n["steps"] == 5: first_half._stop = "sigterm"
    return orig(batch)
first_half._parts = counting
test_fail(lambda: first_half.fit(2), contains="paused (sigterm) at step 5/")
second = tiny_gold(balance=True).resume_from(ck); second.fit(2)
# the same batches and sampling give the same losses; the weights agree to float noise (CPU kernels sum in an order that varies a little)
test_eq([x["loss"] for x in second.log if "loss" in x][-1], [x["loss"] for x in whole.log if "loss" in x][-1])
same = all(torch.allclose(a, b, atol=5e-4) for (_, a), (_, b) in zip(whole.trainable(), second.trainable()))
print("resumed weights equal the uninterrupted run's:", same)
assert same and whole.state["scales"].keys() == second.state["scales"].keys()
test_close([whole.state["scales"][k] for k in whole.state["scales"]], [second.state["scales"][k] for k in whole.state["scales"]], eps=1e-4)

resumed weights equal the uninterrupted run's: True


The same records train an adapter: its matrices and the heads it carries learn, the model under it stays as it was, and switching the adapter off gives it back exactly.

In [ ]:
base_model = load_extractor(FIX / "base")
before = {n: p.detach().clone() for n, p in base_model.named_parameters()}
torch.manual_seed(0)
ad = GlinerLearner(adapter_model(base_model, r=4), train, train[:4], bs=4, lora_lr=1e-2, head_lr=1e-3, warmup=0.0, verbose=False, max_tokens=None)
print(ad)
ad.fit(2)
test_eq(Counter(_group(n) for n, _ in ad.trainable()).keys() >= {"lora", "heads"}, True)
plain = lambda n: n.replace("base_model.model.", "").replace(".base_layer", "").replace(".original_module", "")
kept = {plain(n): p for n, p in ad.model.named_parameters() if "lora_" not in n and "modules_to_save" not in n}
assert len(kept) > 20 and all(torch.equal(before[k], p) for k, p in kept.items())
ad.model.eval()
with ad.model.disable_adapter(): off = decision_logits(ad.model, [row])
test_eq(off, decision_logits(load_extractor(FIX / "base"), [row]))

GlinerLearner(11 train · 4 valid · adapter · bs 4 · fp32 · cpu)


## Saving the model

A full fine-tune is saved as a GLiNER2 checkpoint, which `AutoExtractor.from_pretrained` loads like the base; an adapter in PEFT's format, which loads over its base. Either way `training.json` records how it was made.

In [ ]:
#| export
def gold_record(learn:GlinerLearner, **extra) -> dict:
    "What trained a model: the code, the environment and hardware, the run's settings, the time it took and its log; plus `extra` (data, scores)"
    from sysone.record import environment, hardware, git_info, entry_point
    import sysone, datetime, os
    shipped = json.loads(os.environ.get("SYSONE_SOURCE") or "{}")
    return {"recorded": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "code": {"git": git_info() or shipped.get("git"), "sysone": sysone.__version__, "entry": entry_point()},
            "environment": environment(), "hardware": hardware(), "job": json.loads(os.environ.get("SYSONE_JOB") or "null"),
            "training": {"kind": "adapter" if hasattr(learn.model, "peft_config") else "full", "epochs": learn.state.get("epochs"),
                         "steps": learn.state["step"], "bs": learn.bs, "lr": learn.enc_lr, "head_lr": learn.head_lr, "lora_lr": learn.lora_lr,
                         "weight_decay": learn.weight_decay, "weights": learn.weights, "balance": learn.balance, "sampling": learn.sampling,
                         "precision": learn.precision,
                         "seed": learn.seed, "max_tokens": learn.max_tokens, "max_len": learn.max_len, "seconds": round(learn.state["seconds"], 1),
                         "train": len(learn.train), "valid": len(learn.valid), "trainable": sum(p.numel() for _, p in learn.trainable()),
                         "resumed_from_step": learn.state.get("resumed_from_step"), "log": learn.log}} | extra

def export_model(model, path, base:str=BASE, record:dict|None=None, name:str|None=None) -> Path:
    "The trained model in `path`: a GLiNER2 checkpoint, or an adapter in PEFT's format; `sysone_gliner.json` gives its name and the model it was trained from, `training.json` how"
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    adapter = hasattr(model, "peft_config")
    with warnings.catch_warnings():               # PEFT takes the adapter's 1-D bias vectors (lora_B.bias) for unsharded ZeRO-3 tensors; they are whole
        warnings.filterwarnings("ignore", message=r".*LoRA tensor\(s\) have invalid shape.*")
        (model if adapter else _unwrap(model)).save_pretrained(str(path))
    (path / "sysone_gliner.json").write_text(json.dumps({"backend": "gliner2", "base": base, "adapter": adapter, "regime": "multitask gold labels",
                                                          "temperature": {}, "name": name or path.name}, indent=2))
    if record is not None: (path / "training.json").write_text(json.dumps(record, indent=1, default=str))
    return path

In [ ]:
out = export_model(learn.model, Path(tempfile.mkdtemp()) / "tiny-multi", record=gold_record(learn, data={"rows": len(train)}))
print(sorted(p.name for p in out.iterdir()))
again = AutoExtractor.from_pretrained(str(out))
test_eq(decision_logits(again, [row]), decision_logits(learn.model, [row]))
test_eq(json.loads((out / "training.json").read_text())["training"]["kind"], "full")
test_eq(json.loads((out / "sysone_gliner.json").read_text())["adapter"], False)
aout = export_model(ad.model, Path(tempfile.mkdtemp()) / "tiny-adapter", record={})
assert (aout / "adapter_model.safetensors").exists() and json.loads((aout / "sysone_gliner.json").read_text())["adapter"]
reloaded = PeftModel.from_pretrained(load_extractor(FIX / "base"), str(aout)).eval()
test_eq(decision_logits(reloaded, [row]), decision_logits(ad.model, [row]))       # the adapter, its bias vectors and its heads, back whole

['config.json', 'encoder_config', 'model.safetensors', 'sysone_gliner.json', 'tokenizer.json', 'tokenizer_config.json', 'training.json']


A model goes to the Hub with a card: `model_card` writes one from the export's own record (what it was trained on and how) and the scores you give it, with a snippet that loads it, and `push_model` uploads the folder, private by default. With `replace`, the repo ends up holding exactly the folder's files (what it held before stays in its history).

In [ ]:
#| export
def _md_table(rows:list, cols:list) -> str:
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + ["| " + " | ".join(map(str, r)) + " |" for r in rows])

def model_card(path, repo_id:str, base:str=BASE, scores:dict|None=None, limitations:list|None=None, data_repo:str|None=None,
               description:str|None=None, thresholds:dict|None=None, license:str="other", scores_note:str|None=None) -> str:
    "README.md for an export of `GlinerLearner` (an adapter or a whole model): what it does, how to load and ask it, its `scores` ({model: {column: value}}), limitations, data and training"
    path = Path(path); rec = json.loads((path / "training.json").read_text()) if (path / "training.json").exists() else {}
    tr, data = rec.get("training") or {}, rec.get("data") or {}
    adapter = (path / "adapter_config.json").exists()
    rank = json.loads((path / "adapter_config.json").read_text()).get("r") if adapter else None
    th = thresholds or {"entities": 0.5, "relations": 0.3, "structures": 0.3}
    name = repo_id.split("/")[-1]
    load = (f'model = PeftModel.from_pretrained(AutoExtractor.from_pretrained("{base}"), "{repo_id}").get_base_model()' if adapter
            else f'model = AutoExtractor.from_pretrained("{repo_id}")')
    front = ["---", f"license: {license}", f"base_model: {base}", *(["library_name: peft"] if adapter else []), "pipeline_tag: schema-extraction",
             "tags:", *[f"- {t}" for t in ("gliner2", *(("lora",) if adapter else ()), "information-extraction", "relation-extraction", "event-extraction",
                                          "causal-relations", "decision-model", "sysone")],
             "language:", "- en", "---", ""]
    body = [f"# {name}", "",
            description or (f"[{base}](https://huggingface.co/{base}) trained further on gold labels from public datasets, "
                            + ("as a LoRA adapter, " if adapter else "") + "to make decisions of every kind, find entities of any type, and relate them: "
                            "entity to entity, event to event (causes, is a precondition of, happens before, has subevent), and statement to statement "
                            "(implies, rules out). Trained with [sysonelib](https://github.com/sgaseretto/sysonelib)'s `sysone.multitask`."), "",
            "## Use it", "", "```python", "from gliner2 import AutoExtractor", *(["from peft import PeftModel"] if adapter else []), "", load, "",
            'model.classify_text("My card was charged twice", {"intent": ["billing", "shipping", "account"]})',
            f'model.extract_entities("Ana Ruiz joined Acme in Madrid", ["person", "company", "city"], threshold={th["entities"]})',
            'model.extract_relations("1. Will Bitcoin close above $100,000 on June 30?\\n2. Will Bitcoin close above $90,000 on June 30?",',
            '                        {"implies": "if the first is true, the second is true as well",',
            '                         "rules out": "the two cannot both be true"},' + f" threshold={th['relations']})", "```", "",
            f"It finds relations more cautiously than entities: entities read best at a threshold of {th['entities']}, relations and record fields at "
            f"{th['relations']}. To relate many statements (markets, claims, events), sysone's `statement_graph` reads them a few blocks at a time so "
            "that every two share a list.", ""]
    if adapter:
        body += [f"The adapter is not merged: the base's weights stay as they are. It adds rank-{rank} changes, with their biases, to the encoder's linear "
                 "layers, and carries its own copies of the heads (decisions, spans, records, relations); switched off, it gives the base back. "
                 "PEFT's `merge_and_unload()` folds it into the weights.", ""]
    if scores:
        cols = list(dict.fromkeys(c for v in scores.values() for c in v))
        body += ["## Scores", "", "On held-out test suites: accuracy for decisions, strict F1 (exactly the gold's words) for entities, records and relations."
                 + (f" {scores_note}" if scores_note else ""), "",
                 _md_table([[m] + [("" if v.get(c) is None else f"{v[c]:.3f}") for c in cols] for m, v in scores.items()], ["model", *cols]), ""]
    if limitations: body += ["## Limitations", ""] + [f"- {x}" for x in limitations] + [""]
    if data:
        body += ["## Data", "", f"Training records from {len(data.get('train') or {})} datasets, with their gold labels" + (f", stored as [{data_repo}](https://huggingface.co/datasets/{data_repo})" if data_repo else "")
                 + ". Each source's own terms apply to what came from it; some (GPL-3.0, CC BY-SA 4.0) ask that derived work share alike, hence `license: other`.", "",
                 _md_table(sorted(((k, f"{v:,}") for k, v in (data.get("train") or {}).items()), key=lambda r: -int(r[1].replace(",", ""))), ["dataset", "training records"]), ""]
        srcs = data.get("sources") or {}
        if srcs: body += [_md_table([(k, v.get("repo", ""), v.get("license", "")) for k, v in srcs.items()], ["source", "from", "terms"]), ""]
    if tr:
        gpu = ", ".join(((rec.get("hardware") or {}).get("accelerator") or {}).get("devices") or ["?"])
        body += ["## Training", "",
                 f"- {'A LoRA adapter, rank ' + str(rank) if adapter else 'Every weight'}; {tr.get('trainable', 0):,} trainable numbers",
                 f"- {tr.get('train', 0):,} training records, {tr.get('epochs')} epochs, {tr.get('steps', 0):,} steps of up to {tr.get('bs')} records and "
                 f"{tr.get('max_tokens')} padded tokens; {tr.get('precision')}",
                 f"- Learning rates: {tr.get('lora_lr') if adapter else tr.get('lr')} ({'LoRA' if adapter else 'encoder'}), {tr.get('head_lr')} (heads); AdamW, "
                 f"weight decay {tr.get('weight_decay')}, warmup then cosine decay",
                 f"- Loss: GLiNER2's per-head losses (decisions, relations, spans)"
                 + (", each divided by its running average" if tr.get("balance") else "") + ", weighted " + ", ".join(f"{k} {v:g}" for k, v in (tr.get("weights") or {}).items()),
                 f"- {tr.get('seconds', 0) / 60:.0f} minutes on {gpu}", ""]
    (path / "README.md").write_text("\n".join(front + body))
    return "\n".join(front + body)

def push_model(path, repo_id:str, private:bool=True, replace:bool=False, commit_message:str|None=None, token=None, **card_kw) -> dict:
    "Write the card and upload an export of `GlinerLearner` to a Hub model repo, private by default; with `replace`, files the folder lacks are removed from the repo"
    from huggingface_hub import HfApi
    model_card(path, repo_id, **card_kw)
    api = HfApi(token=token)
    api.create_repo(repo_id, private=private, exist_ok=True)
    info = api.upload_folder(repo_id=repo_id, folder_path=str(path), commit_message=commit_message or "Model, card and training record",
                             delete_patterns="*" if replace else None)
    return {"repo": repo_id, "private": private, "commit": info.oid, "url": info.commit_url}

In [ ]:
card = model_card(aout, "me/tiny-multi", scores={"base": {"REDFM relations": 0.18}, "this adapter": {"REDFM relations": 0.48}},
                  limitations=["Relations between events stay weak."], data_repo="me/tiny-mix")
assert "library_name: peft" in card and "| this adapter | 0.480 |" in card and "relations and record fields at 0.3" in card
model = PeftModel.from_pretrained(AutoExtractor.from_pretrained(str(FIX / "base")), str(aout)).get_base_model()     # the card's snippet, on the fixture
assert "intent" in model.classify_text("My card was charged twice", {"intent": ["billing", "shipping", "account"]})
assert "entities" in model.extract_entities("Ana Ruiz joined Acme in Madrid", ["person", "company", "city"], threshold=0.5)
assert "relation_extraction" in model.extract_relations("1. Will it rain?\n2. Will it pour?", {"implies": "if the first is true, the second is true as well"}, threshold=0.3)

## Keeping the data

Building the mix takes a Kaggle job of twenty minutes or more, so it is worth keeping. `gold_data_card` writes a dataset card for a built folder (`train.jsonl.gz`, `valid.jsonl.gz`, `data.json`, the test rows under `eval/`) from its own `data.json`: what it holds, where each part came from and under which terms, and how to read it. The folder can then go to the Hub with `push_data` (a private dataset repo, which Modal jobs read as `hf:owner/name`) and to Kaggle with `kaggle_dataset` (which Kaggle jobs mount as `kaggle:owner/name`).

In [ ]:
#| export
def gold_data_card(path, repo_id:str, title:str|None=None, built_by:str|None=None) -> Path:
    "Write README.md, a dataset card, for a built gold mix in `path`, from its data.json: files, the mix, the test suites, the sources and their terms, the record format, and how to read it"
    path = Path(path); d = json.loads((path / "data.json").read_text())
    files = sorted(str(p.relative_to(path)) for p in path.rglob("*") if p.is_file() and p.name != "README.md")
    mb = lambda f: (lambda n: f"{n / 1e6:.1f} MB" if n >= 1e5 else f"{max(1, round(n / 1e3))} KB")((path / f).stat().st_size)
    table = lambda rows, cols: "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + ["| " + " | ".join(map(str, r)) + " |" for r in rows])
    sources = d.get("sources") or {}
    out = ["---", "license: other", f"pretty_name: {title or repo_id.split('/')[-1]}", "viewer: false",
           "task_categories:", "- token-classification", "- text-classification", "- zero-shot-classification",
           "tags:", "- gliner2", "- information-extraction", "- relation-extraction", "- event-relations", "- sysone", "---", "",
           f"# {title or repo_id.split('/')[-1]}", "",
           f"GLiNER2 training records with gold labels for decisions, entities and relations (between entities, events and statements), "
           f"{d.get('train_rows', 0):,} for training and {d.get('valid_rows', 0):,} for validation, and {sum((d.get('eval') or {}).values()):,} test rows "
           f"in {len(d.get('eval') or {})} suites, built with [sysone](https://github.com/sgaseretto/sysonelib)'s `sysone.multitask` readers"
           + (f" by {built_by}" if built_by else "") + ".", "",
           "## Files", "", table([(f"`{f}`", mb(f)) for f in files], ["file", "size"]), "",
           "A copy on Kaggle holds the same files, the `.gz` ones decompressed (Kaggle unpacks them).", "",
           "## The training mix", "", table(sorted(((k, f"{v:,}") for k, v in (d.get("train") or {}).items()), key=lambda r: -int(r[1].replace(",", ""))),
                                           ["dataset", "records"]), "",
           f"Dropped before training: {(d.get('dropped') or {}).get('exact', 0):,} rows whose text is a test row's, {(d.get('dropped') or {}).get('near', 0):,} near duplicates "
           f"of one (MinHash, Jaccard ≥ 0.6), and the records GLiNER2's processor refused: {', '.join(f'{k} ({v:,})' for k, v in ((d.get('dropped') or {}).get('refused') or {}).items()) or 'none'}.", "",
           "## Test suites", "", table([(k, f"{v:,}") for k, v in (d.get("eval") or {}).items()], ["suite", "rows"]), "",
           "`eval/rows.jsonl` holds them with their gold; `eval/answers/` and `scores.json`, the answers and scores of the models the build scored.", "",
           "## Sources and terms", "", "The mix gathers sources under different terms; each source's own terms apply to what comes from it.", "",
           table([(k, v.get("repo", ""), v.get("license", "")) for k, v in sources.items()], ["source", "from", "terms"]), "",
           "## Format", "",
           "One JSON object a line. A training record is GLiNER2's: `input` (the text) and `output` (what to find, with the gold), each gold span "
           "`{\"text\", \"start\", \"end\"}` in characters of the text; a relation or type asked with nothing found is a negative. Beside them: "
           "`absent_relations` (relation types asked and absent), `soft` (a decision's gold distribution, when annotators disagreed), `length` "
           "(a rough token count), and `id`, `kind`, `source`, `dataset`.", "",
           "## Reading it", "", "```python", "import gzip, json", "from huggingface_hub import snapshot_download", "",
           f'folder = snapshot_download("{repo_id}", repo_type="dataset")',
           'train = [json.loads(l) for l in gzip.open(f"{folder}/train.jsonl.gz", "rt")]', "```", "",
           "sysone's `GlinerLearner` trains on these lines as they are (`sysone.multitask`)."]
    (path / "README.md").write_text("\n".join(out) + "\n")
    return path / "README.md"

In [ ]:
mixdir = Path(tempfile.mkdtemp()) / "mix"; (mixdir / "eval").mkdir(parents=True)
(mixdir / "data.json").write_text(json.dumps({"train": {"REDFM": 2, "MAVEN-ERE": 1}, "train_rows": 3, "valid_rows": 1, "eval": {"redfm": 1},
                                              "sources": {"redfm": {"repo": "Babelscape/REDFM", "license": "CC BY-SA 4.0"}}, "dropped": {"exact": 0, "near": 1, "refused": {}}}))
with gzip.open(mixdir / "train.jsonl.gz", "wt") as f: f.write("\n".join(train[:3]) + "\n")
(mixdir / "eval" / "rows.jsonl").write_text(json.dumps(row) + "\n")
card = gold_data_card(mixdir, "me/tiny-mix").read_text()
assert "| REDFM | 2 |" in card and "| redfm | Babelscape/REDFM | CC BY-SA 4.0 |" in card and "`eval/rows.jsonl`" in card and card.startswith("---\nlicense: other")
print(card[:600])

---
license: other
pretty_name: tiny-mix
viewer: false
task_categories:
- token-classification
- text-classification
- zero-shot-classification
tags:
- gliner2
- information-extraction
- relation-extraction
- event-relations
- sysone
---

# tiny-mix

GLiNER2 training records with gold labels for decisions, entities and relations (between entities, events and statements), 3 for training and 1 for validation, and 1 test rows in 1 suites, built with [sysone](https://github.com/sgaseretto/sysonelib)'s `sysone.multitask` readers.

## Files

| file | size |
|---|---|
| `data.json` | 1 KB |
| `eval/r


## Answers and scores

`answers` asks a model each row's schema and keeps what it finds with its confidence and its offsets; decisions as the logits of every label. `gold_scores` compares them with the gold, per suite of rows:

- **Decisions**: accuracy (the top label against the gold's top label) and calibration error.
- **Entities, relations, record fields**: precision, recall and F1, *strict* (the same type and the same text, after lowercasing and collapsing spaces) and *relaxed* (one text contains the other: a span one word longer or shorter still counts, which is fair to gold that marks whole clauses). Relations are also scored per family when given one, so causal, temporal, subevent and coreference relations each get their own F1.
- **Confidence intervals**: 95% intervals from resampling the suite's rows 1,000 times, so that a difference between two models can be read against the noise of a few hundred rows.

In [ ]:
#| export
def _text(x) -> str: return x["text"] if isinstance(x, dict) else str(x)

def _found(ask:dict, res:dict) -> dict:
    "A batch_extract result as {entities, relations, structures}, each span [text, confidence, start, end]"
    sp = lambda x: [x.get("text"), round(float(x.get("confidence", 1.0)), 4), x.get("start"), x.get("end")]
    many = lambda v: [sp(x) for x in (v if isinstance(v, list) else [v]) if isinstance(x, dict) and x.get("text")]
    out = {}
    if ask.get("entities"): out["entities"] = {t: many((res.get("entities") or {}).get(t) or []) for t in ask["entities"]}
    if ask.get("relations"):
        rel = res.get("relation_extraction") or {}
        out["relations"] = {n: [sp(p["head"])[:1] + sp(p["tail"])[:1] + [round(float(p.get("confidence", min(p["head"].get("confidence", 1), p["tail"].get("confidence", 1)))), 4)]
                                for p in rel.get(n) or [] if isinstance(p, dict) and p.get("head") and p.get("tail")] for n in ask["relations"]}
    if ask.get("structures"):
        out["structures"] = {n: [{f: many(inst.get(f) or []) for f in fields} for inst in res.get(n) or [] if isinstance(inst, dict)]
                             for n, fields in ask["structures"].items()}
    return out

def answers(model, rows:list, batch_size:int=8, threshold:float=0.5, max_tokens:int|None=8192, thresholds:dict|None=None) -> list:
    "What `model` answers to each row: entities, relations and record fields with confidences and offsets, and each decision's logits; `thresholds` ({'entities': .., 'relations': .., 'structures': ..}) overrides `threshold` for a kind"
    m = _unwrap(model).eval(); out = [{} for _ in rows]
    ext = [i for i, r in enumerate(rows) if any(r["ask"].get(k) for k in ("entities", "relations", "structures"))]
    with torch.inference_mode():
        for b in length_batches([row_length(rows[i]) for i in ext], batch_size, max_tokens):
            chunk = [ext[k] for k in b]
            res = m.batch_extract([rows[i]["text"] for i in chunk], [ask_schema(rows[i]["ask"], thresholds) for i in chunk], batch_size=len(chunk),
                                  threshold=threshold, include_confidence=True, include_spans=True)
            for i, x in zip(chunk, res): out[i] = _found(rows[i]["ask"], x)
    dec = [i for i, r in enumerate(rows) if r["ask"].get("classifications")]
    for i, lg in zip(dec, decision_logits(m, [rows[i] for i in dec], batch_size, max_tokens)): out[i]["logits"] = lg
    return out

In [ ]:
#| export
def _items(kind:str, d:dict, gold:bool) -> set:
    "A row's gold or answers of one kind as a set of (label, text, …), texts normalised"
    if kind == "entities":
        return {(t, _norm(_text(x if gold else x[0]))) for t, xs in (d.get("entities") or {}).items() for x in (_span_list(xs) if gold else xs)}
    if kind == "relations":
        return {(n, _norm(_text(p[0])), _norm(_text(p[1]))) for n, ps in (d.get("relations") or {}).items() for p in ps}
    return {((n, f), _norm(_text(x if gold else x[0]))) for n, insts in (d.get("structures") or {}).items() for i in insts
            for f, xs in i.items() for x in (_span_list(xs) if gold else xs)}

def _close(a:str, b:str) -> bool: return a == b or (len(a) > 0 and len(b) > 0 and (a in b or b in a))

def _hits(pred:set, gold:set, relaxed:bool) -> int:
    "How many answers match a gold item: the same label and texts, or (relaxed) texts that contain one another; each gold item matched once"
    if not relaxed: return len(pred & gold)
    used, tp = set(), len(pred & gold)
    used |= pred & gold
    for p in pred - gold:
        g = next((g for g in gold - used if g[0] == p[0] and all(_close(x, y) for x, y in zip(p[1:], g[1:]))), None)
        if g is not None: used.add(g); tp += 1
    return tp

KINDS = {"entities": "entities", "relations": "relations", "records": "structures"}

def row_counts(pred:dict, row:dict, families:dict|None=None) -> dict:
    "One row's counts: decisions (right, asked, confidence) and, per kind of span and per relation family, (hits, answers, gold), strict and relaxed"
    out, gold, ask = {}, row.get("gold") or {}, row["ask"]
    for task, lg in (pred.get("logits") or {}).items():
        g = (gold.get("classifications") or {}).get(task)
        if g is None: continue
        g = max(g, key=g.get) if isinstance(g, dict) else g
        labels = list(lg); p = _softmax([lg[l] for l in labels])
        out.setdefault("decisions", []).append((labels[int(p.argmax())] == g, float(p.max())))
    for kind, key in KINDS.items():
        if not ask.get(key): continue
        P, G = _items(kind, pred, False), _items(kind, gold, True)
        groups = {kind: (P, G)}
        if kind == "relations" and families:
            for f in _unique(families.get(n) for n in ask["relations"] if families.get(n)):
                groups[f"relations:{f}"] = ({x for x in P if families.get(x[0]) == f}, {x for x in G if families.get(x[0]) == f})
        for name, (p, g) in groups.items():
            for relaxed in (False, True): out[name + ("~" if relaxed else "")] = (_hits(p, g, relaxed), len(p), len(g))
    return out

def _boot(stat, arr:np.ndarray, n:int, seed:int) -> list:
    "A 95% interval for `stat` from resampling the rows of `arr` `n` times; resamples where it is undefined are left out"
    rng = np.random.default_rng(seed)
    vals = np.array([stat(arr[rng.integers(0, len(arr), len(arr))]) for _ in range(n)], float)
    vals = vals[np.isfinite(vals)]
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)] if len(vals) else [float("nan")] * 2

def _prf(c) -> tuple:
    tp, npred, ngold = map(float, c)
    p, r = (tp / npred if npred else float("nan")), (tp / ngold if ngold else float("nan"))
    return p, r, float(_f1(tp, npred - tp, ngold - tp))

def gold_scores(pred:list, rows:list, by:str="suite", families:dict|None=None, boot:int=1000, seed:int=0) -> dict:
    "Scores against the gold per suite (the rows' `by`): decision accuracy and calibration, span precision, recall and F1 (strict, and relaxed with `~`), with 95% bootstrap intervals over rows"
    counts = [row_counts(p, r, families) for p, r in zip(pred, rows)]
    groups = defaultdict(list)
    for i, r in enumerate(rows): groups[r.get(by) or r.get("source")].append(i)
    out = {}
    for g, ix in groups.items():
        s = {"rows": len(ix)}
        dec = [(i, x) for i in ix for x in counts[i].get("decisions", [])]
        if dec:
            right = np.array([x[0] for _, x in dec], float); conf = [x[1] for _, x in dec]
            per = np.array([[sum(x[0] for x in counts[i].get("decisions", [])), len(counts[i].get("decisions", []))] for i in ix], float)
            s |= {"decisions": len(dec), "accuracy": float(right.mean()), "accuracy_ci": _boot(lambda a: a[:, 0].sum() / max(1, a[:, 1].sum()), per, boot, seed),
                  "ece": ece(conf, right)}
        for name in _unique(k for i in ix for k in counts[i] if k != "decisions"):
            per = np.array([counts[i].get(name, (0, 0, 0)) for i in ix], float)
            p, r, f = _prf(per.sum(0))
            s |= {f"{name}_p": p, f"{name}_r": r, f"{name}_f1": f, f"{name}_n": int(per[:, 2].sum())}
            if not name.endswith("~"): s[f"{name}_f1_ci"] = _boot(lambda a: _prf(a.sum(0))[2], per, boot, seed)
        out[g] = s
    return out

In [ ]:
pred = [{"entities": {"event": [["storm", 0.9, 4, 9], ["flood", 0.8, 32, 37], ["the coast", 0.6, 14, 23]], "person": []},
         "relations": {"causes": [["storm", "flood", 0.7], ["the flood", "destroyed", 0.6]], "happens before": []},
         "structures": {"Destroying": [{"trigger": [["destroyed", 0.9, 38, 47]], "Patient": [["bridge", 0.5, 52, 58]]}]},
         "logits": {"tone": {"calm": -1.0, "alarming": 1.0}}}]
sc = gold_scores(pred, [row | {"suite": "demo"}], boot=200)["demo"]
test_eq(sc["accuracy"], 1.0)
test_eq((sc["entities_p"], sc["entities_r"]), (2 / 3, 2 / 4))              # storm and flood found; destroyed and ended missed; "the coast" wrong
test_eq(sc["relations_f1"], 0.5)                                            # "the flood" is not "flood" ...
test_eq(sc["relations~_f1"], 1.0)                                           # ... unless one text may contain the other
test_eq(sc["records~_f1"], 1.0)
fam = gold_scores([{"relations": {"causes": [["storm", "flood", 1]], "happens before": [["storm", "ended", 1]]}}],
                  [m[0] | {"suite": "maven"}], families=relation_families(), boot=100)["maven"]
test_eq((fam["relations:causal_r"], fam["relations:temporal_r"]), (0.5, 0.5))
sorted(k for k in fam if k.startswith("relations:"))[:6]

['relations:causal_f1',
 'relations:causal_f1_ci',
 'relations:causal_n',
 'relations:causal_p',
 'relations:causal_r',
 'relations:causal~_f1']

In [ ]:
got = answers(learn.model, [row, rows[0]])
assert set(got[0]) == {"entities", "relations", "structures", "logits"} and all(len(x) == 4 for xs in got[0]["entities"].values() for x in xs)
test_eq(list(got[1]["logits"]), ["intent", "anger"])

One threshold need not suit every kind: a trained model can be best at 0.5 for entities and lower for relations. `thresholds` sets a kind's own, over the call's `threshold`:

In [ ]:
n_found = lambda a: {k: sum(len(v) for v in a.get(k, {}).values()) for k in ("entities", "relations")}
low, mixed = answers(learn.model, [row], threshold=0.0)[0], answers(learn.model, [row], threshold=0.0, thresholds={"relations": 1.0})[0]
assert n_found(low)["relations"] > 0 and n_found(mixed)["relations"] == 0 and n_found(mixed)["entities"] == n_found(low)["entities"]

## A dependency graph over many statements

A model reads a list of statements in one pass, but not hundreds: its encoder takes about 512 tokens, some twenty questions. `statement_graph` finds the relations among any number of statements by reading them a few at a time. It cuts them into blocks and reads every pair of blocks together as one list, so every two statements share a list at least once. That takes about (n / block)² / 2 lists: 741 for 305 statements in blocks of 8, against 46,360 pairs one at a time. A relation found in several lists keeps its highest confidence.

In [ ]:
#| export
def cover(n:int, block:int) -> list:
    "Lists of indices in which every pair of `n` items shares a list: the items cut into blocks of `block`, every two blocks read together"
    blocks = [list(range(s, min(s + block, n))) for s in range(0, n, block)]
    return [a + b for k, a in enumerate(blocks) for b in blocks[k + 1:]] or blocks[:1]

def statement_graph(model, statements:list, relations:dict|None=None, block:int=8, threshold:float=0.5, batch_size:int=8) -> list:
    "The relations among `statements` as edges (i, j, relation, confidence), read a few blocks at a time (`cover`), each list numbered as in training; an edge seen in several lists keeps its highest confidence"
    rel = relations or STATEMENT_RELATIONS
    rows, where = [], []
    for g in cover(len(statements), block):
        text, spans = "", []
        for k, i in enumerate(g):
            text += ("\n" if k else "") + f"{k + 1}. "
            s = statements[i].strip(); spans.append((len(text), len(text) + len(s), i)); text += s
        rows.append({"text": text, "ask": {"relations": rel}}); where.append(spans)
    m, edges = _unwrap(model).eval(), {}
    def at(spans, x):
        mid = (x["start"] + x["end"]) / 2 if x.get("start") is not None else None
        return next((i for s, e, i in spans if mid is not None and s <= mid < e), None)
    with torch.inference_mode():
        for b in length_batches([row_length(r) for r in rows], batch_size, 8192):
            res = m.batch_extract([rows[k]["text"] for k in b], [ask_schema(rows[k]["ask"]) for k in b], batch_size=len(b), threshold=threshold,
                                  include_confidence=True, include_spans=True)
            for k, x in zip(b, res):
                for name, ps in (x.get("relation_extraction") or {}).items():
                    for p in ps or []:
                        if not (isinstance(p, dict) and p.get("head") and p.get("tail")): continue
                        h, t = at(where[k], p["head"]), at(where[k], p["tail"])
                        if h is None or t is None or h == t: continue
                        c = float(p.get("confidence", min(p["head"].get("confidence", 1), p["tail"].get("confidence", 1))))
                        edges[(h, t, name)] = max(edges.get((h, t, name), 0.0), c)
    return sorted([(h, t, r, round(c, 4)) for (h, t, r), c in edges.items()], key=lambda e: (-e[3], e[:3]))

In [ ]:
for n_, b_ in [(1, 8), (7, 8), (20, 8), (305, 8)]:
    lists = cover(n_, b_)
    assert all(any(i in l and j in l for l in lists) for i, j in combinations(range(n_), 2)) and max(map(len, lists)) <= 2 * b_
test_eq(len(cover(305, 8)), 741)
qs = ["Will Bitcoin close above $90,000 on June 30, 2025?", "Will Bitcoin close above $80,000 on June 30, 2025?", "Will Arsenal win the Champions League?",
      "Will Arsenal reach the final of the Champions League?", "Will Lula win the 2026 Brazilian presidential election?",
      "Will the Workers' Party candidate win the 2026 Brazilian presidential election?", "Will the US enter a recession by December 31, 2025?"]
edges = statement_graph(learn.model, qs, block=3, threshold=0.0)
assert edges and all(0 <= h < len(qs) and 0 <= t < len(qs) and h != t for h, t, _, _ in edges)
edges[:3]

[(1, 6, 'causes', 0.1424),
 (1, 6, 'implies', 0.1424),
 (1, 6, 'is a precondition of', 0.1424)]

To score a graph, `market_pool` draws many market groups that share no entity (no two about the same election, tournament, match, event, or asset on the same date), so every relation in the pool is known: those inside each group, and none between groups. `graph_scores` compares a graph's edges with them.

In [ ]:
#| export
def market_pool(groups:int, seed:int=0, part:str="test", kinds:tuple=MARKET_KINDS[:-1], world:dict=MARKET_WORLD) -> tuple:
    "Questions from up to `groups` market groups that share no entity, shuffled, and every relation among them as (i, j, relation), a symmetric one both ways"
    rng, used, clauses, pairs, made = random.Random(f"pool:{seed}:{part}"), set(), [], [], 0
    for _ in range(groups * 5):                         # a draw whose entity is taken is skipped; stop at `groups` groups
        if (got := _market(rng.choice(list(kinds)), rng, world, part, used)) is None: continue
        b = len(clauses); clauses += got[0]; pairs += [(i + b, j + b, r) for i, j, r in got[1]]; made += 1
        if made == groups: break
    order = list(range(len(clauses))); rng.shuffle(order); place = {i: k for k, i in enumerate(order)}
    gold = {(place[i], place[j], r) for i, j, r in market_closure(len(clauses), pairs)}
    gold |= {(j, i, r) for i, j, r in gold if r in SYMMETRIC}
    return [f"Will {clauses[i]}?" for i in order], sorted(gold)

def graph_scores(edges:list, gold:list, threshold:float=0.5) -> dict:
    "Precision, recall and F1 of a graph's edges (i, j, relation, confidence) at `threshold` against the gold (i, j, relation), overall and per relation"
    pred = {(h, t, r) for h, t, r, c in edges if c >= threshold}; gold = set(gold)
    out = {}
    for name in [None] + sorted({r for *_, r in gold | pred}):
        p, g = {e for e in pred if name in (None, e[2])}, {e for e in gold if name in (None, e[2])}
        tp = len(p & g)
        out[name or "all"] = {"p": tp / len(p) if p else float("nan"), "r": tp / len(g) if g else float("nan"), "f1": _f1(tp, len(p) - tp, len(g) - tp),
                              "edges": len(p), "gold": len(g)}
    return out

In [ ]:
pool, gold = market_pool(15, seed=0)
print(len(pool), "questions,", len(gold), "relations among them")
assert len(pool) >= 25 and len(gold) >= 10 and all(0 <= i < len(pool) and 0 <= j < len(pool) for i, j, _ in gold)
test_eq(graph_scores([(i, j, r, 0.9) for i, j, r in gold], gold)["all"]["f1"], 1.0)
test_eq(graph_scores([(0, 1, "implies", 0.4)], gold, threshold=0.5)["all"]["edges"], 0)

39 questions, 55 relations among them


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()